# Lab 3-3 · Pipeline ครบวงจร: เตรียมข้อมูล → ปรับ Hyperparameter → สร้างโมเดลทำนาย → ทดลองและประเมินผล (MovieLens)

ต่อยอดจาก Lab 3-2 (ซึ่งเปรียบเทียบเทคนิคด้วย "ค่าตั้งต้น" ของทุกโมเดล) สมุดบันทึกนี้แสดง **ท่อประมวลผล (pipeline) 12 ขั้น**
ที่ระบบจริงต้องมี ตั้งแต่การเตรียมข้อมูล การปรับจูนไฮเพอร์พารามิเตอร์ตาม **ลำดับที่ให้ผลตอบแทนสูงสุดก่อน (ตาราง 3.66)**
บนชุดตรวจสอบเท่านั้น จนถึงการสร้าง **โมเดลทำนายขั้นสุดท้าย** สองแบบ (จัดอันดับด้วย Learning-to-Rank แบบ stacking และ
ทำนายคะแนนด้วย Content-kNN ที่ปรับจูนแล้ว) แล้วประเมินตามบทที่ 2 ทั้ง Accuracy และ Beyond-Accuracy บน Validation และ Test

| ขั้น | Pipeline | หัวข้อในตำรา |
|---|---|---|
| ① | โหลด · ตรวจสอบ · ทำความสะอาดข้อมูล (audit / dedupe / k-core) | 2.8, 3.20.1 ข้อ 1–2 |
| ② | แบ่ง Train / Valid-A / Valid-B / Test ตามเวลาในผู้ใช้ (valid สองชุดกันการ "ปรับให้พอดีกับชุดตรวจสอบ") | 2.8.1, ตาราง 2.10 |
| ③ | สร้างเขตข้อมูลเนื้อหาหลายเขต (ชื่อ · แนว · tag · ทศวรรษ) + น้ำหนักเขตข้อมูล | 3.11.5 (BM25F) |
| ④ | ปรับจูนการแทนเนื้อหา: min_df · ngram · TF/IDF scheme · น้ำหนักเขตข้อมูล (ลำดับ 1–4 ของตาราง 3.66) | 3.4–3.6 |
| ⑤ | ปรับจูนฟังก์ชันจัดอันดับ BM25 (k₁ × b) และคำในข้อคำถาม | 3.11.3 |
| ⑥ | ปรับจูนโปรไฟล์: like_thr · β/γ ของ Rocchio · NB α (ลำดับ 6) | 3.8, 3.12 |
| ⑦ | ปรับจูนมิติแฝง LSA k (ลำดับ 7) | 3.13.2 |
| ⑧ | ปรับจูนการผสาน: น้ำหนัก convex / RRF k (ลำดับ 8) | 3.18.1 |
| ⑨ | ปรับจูนความหลากหลาย λ ของ MMR ภายใต้ guardrail (ลำดับ 9) | 3.10.5 |
| ⑩ | **โมเดลทำนายขั้นสุดท้าย**: LTR stacking (pointwise) จากคะแนนทุกระบบ + ความนิยม + ปี · ปรับเทียบ Platt + ECE | 3.18.2–3.18.3 |
| ⑪ | **โมเดลทำนายคะแนน**: Content-kNN (k · shrink · ปริภูมิ) และ Ridge บนคุณลักษณะ → MAE/RMSE | 2.1 |
| ⑫ | ประเมินครั้งเดียวบน Test · ตาราง/กราฟ Accuracy & Beyond-Accuracy · ตัวอย่าง Top-K · บันทึก artifacts · **GUI** | 2.14 |

> **Colab**: รันทุกเซลล์ตามลำดับ · ดาวน์โหลดข้อมูลอัตโนมัติ · เวลารันราว 6–10 นาที (CPU) · `FAST_MODE=True` → ~2 นาที
> การปรับจูนทั้งหมดใช้ **ผู้ใช้กลุ่มย่อยของ Valid-A** (`N_TUNE` คน) เพื่อประหยัดเวลา — ประกาศไว้ใน Protocol Card

## Part 0 · ติดตั้ง นำเข้า และค่าตั้งค่า

In [ ]:
import importlib, subprocess, sys, os, io, zipfile, urllib.request, json, itertools, time
for pkg in ["ipywidgets"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 180); pd.set_option("display.max_colwidth", 70)

FAST_MODE   = False      # True = ผู้ใช้ 200 คน + กริดเล็กลง
SEED        = 2569
K           = 10
LIKE_THR    = 4.0
DISLIKE_THR = 2.0
K_CORE      = 5
N_TUNE      = 150 if not FAST_MODE else 80     # จำนวนผู้ใช้ (สุ่มจาก Valid-A) ที่ใช้ปรับจูน — ประกาศ
DATA_DIR    = "data/ml-latest-small"
ART_DIR     = "artifacts_lab3_3"               # โฟลเดอร์บันทึกผล/ค่าที่เลือก/โมเดล
os.makedirs(ART_DIR, exist_ok=True)

In [ ]:
# ===================== cbrec_core (ฝัง) — เริ่ม =====================
# -*- coding: utf-8 -*-
"""
==========================================================================================
 cbrec_core.py — แกนกลางของระบบผู้แนะนำอิงเนื้อหา (Content-Based) สำหรับบทที่ 3
                 + ตัวชี้วัดการประเมินผล (Accuracy / Beyond-Accuracy) ของบทที่ 2
                 + เครื่องมือออกแบบการทดลองแบบออฟไลน์ (Offline Experiment)
==========================================================================================
ไฟล์นี้ถูก "ฝัง" อยู่ในสมุดบันทึกทุกเล่ม (Lab3_1 / Lab3_2 / Lab3_3) เพื่อให้แต่ละเล่มรันบน
Google Colab ได้โดยอัปโหลดไฟล์เดียว และยังบันทึกแยกเป็น cbrec_core.py เพื่อ import ซ้ำได้

โครงสร้าง (เรียงตามลำดับของบทที่ 3 → บทที่ 2)
  ส่วนที่ 1  การเตรียมข้อความ (3.4–3.5)   : tokenize · คำหยุด · n-gram
  ส่วนที่ 2  การถ่วงน้ำหนักคำ (3.6–3.7)   : TF 5 รูปแบบ · IDF หลายสูตร · L2 · โคไซน์
  ส่วนที่ 3  BM25 / BM25+ (3.11)          : การอิ่มตัวและปรับความยาวเอกสาร
  ส่วนที่ 4  ปริภูมิความหมาย (3.13)         : LSA (TruncatedSVD) · Sentence-Embedding (ทางเลือก)
  ส่วนที่ 5  โปรไฟล์ผู้ใช้ (3.8, 3.12)      : centroid · ถ่วงคะแนน · Rocchio · Naive Bayes · Logistic
  ส่วนที่ 6  ฟังก์ชันจับคู่และจัดอันดับ (3.9–3.10) : โคไซน์ · Item-kNN · Top-K · MMR
  ส่วนที่ 7  ผสานคะแนนและปรับเทียบ (3.18)  : convex · RRF · Platt · ECE
  ส่วนที่ 8  โมเดลทำนายคะแนน (Rating)      : ค่าเฉลี่ย · Content-kNN regression
  ส่วนที่ 9  ตัวชี้วัดบทที่ 2               : MAE/RMSE · P/R/F1/HR@K · MRR · MAP · nDCG · AUC
                                            Coverage · Gini · LongTail · ILD · Novelty · Calibration · Fairness
  ส่วนที่ 10 การออกแบบการทดลองออฟไลน์ (2.7–2.10, 2.14) : split 4 แบบ · leakage check · k-core ·
                                            candidate set · seed variance · paired bootstrap · Holm · ProtocolCard
  ส่วนที่ 11 การนำเสนอผลและ GUI           : ตารางเทียบโมเดล · กราฟ · รายงานผู้ใช้ · ฟอร์ม ipywidgets

หมายเหตุด้านการทำซ้ำ (Reproducibility)
  * ทุกฟังก์ชันที่มีการสุ่มรับพารามิเตอร์ seed และใช้ numpy.random.default_rng(seed)
  * สูตรตั้งต้นทุกตัวตรงกับที่ใช้คำนวณตัวเลขในบทที่ 3 (TF = 1+log2 f, IDF = ln((1+N)/(1+df))+1, L2)
    หากเปลี่ยนสูตร ตัวเลขจะเปลี่ยนทันที — ต้อง "ประกาศการตั้งค่า" ทุกครั้งที่รายงานผล (ตาราง 3.3)
"""
import math
import re
import time
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp

warnings.filterwarnings("ignore")

SEED = 2569  # ค่าตั้งต้นของ seed ทั้งชุด (พ.ศ. 2569) — เปลี่ยนแล้วผลเปลี่ยน แต่ข้อสรุปเชิงแนวคิดควรคงเดิม


def set_seed(seed=SEED):
    """ตั้ง seed ของ numpy และ random (และ torch หากมี) ให้รันซ้ำได้ผลเดิม"""
    import random
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
    except Exception:
        pass
    return np.random.default_rng(seed)


class Timer:
    """ตัวจับเวลาอย่างง่าย: with Timer('ชื่อ'): ...  → พิมพ์เวลาที่ใช้ (ใช้ดูต้นทุนของแต่ละขั้น)"""
    def __init__(self, label=""):
        self.label = label
    def __enter__(self):
        self.t0 = time.time(); return self
    def __exit__(self, *a):
        self.sec = time.time() - self.t0
        print(f"[เวลา] {self.label}: {self.sec:.2f} วินาที")


# =========================================================================================
# ส่วนที่ 1  การเตรียมข้อความ (หัวข้อ 3.4–3.5)
# =========================================================================================
# รายการคำหยุดชุดเดียวกับที่ใช้คำนวณตัวเลขทุกตัวในบทที่ 3 (ตาราง 3.3: 457 → 263 โทเคน, 249 → 203 คำ)
# ข้อสังเกต: รายการนี้ "ตั้งใจ" เก็บคำปฏิเสธบางคำ (เช่น not, no) ไว้ในรายการคำหยุดตามต้นฉบับของบท
#           หากนำไปใช้กับงานวิเคราะห์ความรู้สึก ต้องนำ not/no/never ออกจากรายการ (หัวข้อ 3.4.6)
STOP_EN = set("""
a an the and or of to in on at for with from as is are was were be been being am
his her their its it he she they them him us we our you your i my me
who which that this these those what when where why how whom whose
not no nor but by into across over under out up down off above below through
while after before during until since again once here there
must can could may might will would shall should do does did have has had
so such than then too very just only own each every all both few more most other same
one two another between within
""".split())


def tokenize(text, keep_thai=False):
    """ตัดคำแบบง่ายที่สุด: แปลงเป็นตัวพิมพ์เล็กแล้วเก็บเฉพาะสายอักษรละติน [a-z]+
    (ตรงกับกฎที่ใช้ในบทที่ 3 ทำให้ "world." → world, "job:" → job, "Mad Max: Fury Road" → mad max fury road)
    keep_thai=True จะเก็บกลุ่มอักขระไทยไว้ด้วย (ยังไม่ตัดคำไทย — ต้องใช้ pythainlp ในหัวข้อ 3.5)
    """
    text = re.sub(r"'s\b", "", text.lower())     # ตัด 's แสดงความเป็นเจ้าของ (potter's → potter) กันโทเคน "s" ลอย
    if keep_thai:
        return re.findall(r"[a-z]+|[฀-๿]+", text)
    return re.findall(r"[a-z]+", text)


def preprocess(text, stop=STOP_EN, ngram_range=(1, 1), keep_thai=False, stemmer=None):
    """กระบวนงานเตรียมข้อความ (ภาพที่ 3.5): ตัดคำ → ตัดคำหยุด → (ลดรูปคำ) → (เพิ่ม n-gram)
    คืนรายการโทเคน (หน่วยที่จะกลายเป็น "มิติ" ของเวกเตอร์)
    * stemmer: ฟังก์ชัน str→str (เช่น nltk PorterStemmer().stem) ค่าตั้งต้น None = ไม่ลดรูปคำ
    * ngram_range=(1,2) จะเพิ่ม bigram ในรูป "w1_w2" ต่อท้าย unigram (ตาราง 3.11)
    """
    toks = [t for t in tokenize(text, keep_thai) if t not in stop]
    if stemmer is not None:
        toks = [stemmer(t) for t in toks]
    lo, hi = ngram_range
    out = []
    for n in range(lo, hi + 1):
        if n == 1:
            out.extend(toks)
        else:
            out.extend("_".join(toks[i:i + n]) for i in range(len(toks) - n + 1))
    return out


def corpus_stats(token_docs):
    """สถิติของคลัง (ตาราง 3.3): จำนวนเอกสาร โทเคนรวม ขนาดคลังศัพท์ ความยาวเฉลี่ย avgdl"""
    n_tok = sum(len(d) for d in token_docs)
    vocab = set(t for d in token_docs for t in d)
    return {"N (เอกสาร)": len(token_docs), "โทเคนรวม": n_tok, "ขนาดคลังศัพท์": len(vocab),
            "avgdl": n_tok / max(1, len(token_docs))}


# =========================================================================================
# ส่วนที่ 2  การถ่วงน้ำหนักคำ TF / IDF / L2 และโคไซน์ (หัวข้อ 3.6–3.7)
# =========================================================================================
def tf_weight(f, scheme="log", max_f=1, doc_len=1):
    """TF ห้ารูปแบบ (ตาราง 3.20)
    raw    : f                      นับตรง ๆ (คำที่ซ้ำ 3 ครั้งได้ 3 เท่า — ไม่มีเพดาน)
    log    : 1 + log2(f)            ลดทอนแบบลอการิทึม (ค่าตั้งต้นของบท; sublinear_tf)
    binary : 1 ถ้า f>0              สนใจแค่ "มี/ไม่มี"
    aug    : 0.5 + 0.5 f/max_f      ปรับตามคำที่ถี่ที่สุดในเอกสาร (กันเอกสารยาวได้เปรียบ)
    norm   : f / doc_len            สัดส่วนต่อความยาวเอกสาร
    """
    if f <= 0:
        return 0.0
    if scheme == "raw":
        return float(f)
    if scheme == "log":
        return 1.0 + math.log2(f)
    if scheme == "binary":
        return 1.0
    if scheme == "aug":
        return 0.5 + 0.5 * f / max(1, max_f)
    if scheme == "norm":
        return f / max(1, doc_len)
    raise ValueError(scheme)


def idf_weight(df, N, scheme="smooth"):
    """IDF หลายสูตร (ตาราง 3.24 — TF-IDF เป็น "ตระกูล" ของสูตร ไม่ใช่สูตรเดียว)
    smooth   : ln((1+N)/(1+df)) + 1   ค่าตั้งต้นของบท (เหมือน sklearn smooth_idf) — ไม่มีคำใดได้ 0
    standard : ln(N/df)               สูตรดั้งเดิม คำที่พบทุกเอกสารได้ 0 พอดี
    prob     : max(0, ln((N-df)/df))  แบบความน่าจะเป็น
    bm25     : ln((N-df+0.5)/(df+0.5) + 1)  สูตรที่ BM25 (Lucene) ใช้
    none     : 1                      ไม่ถ่วง IDF (เท่ากับ TF อย่างเดียว)
    """
    if scheme == "smooth":
        return math.log((1 + N) / (1 + df)) + 1.0
    if scheme == "standard":
        return math.log(N / df)
    if scheme == "prob":
        return max(0.0, math.log((N - df) / df)) if df < N else 0.0
    if scheme == "bm25":
        return math.log((N - df + 0.5) / (df + 0.5) + 1.0)
    if scheme == "none":
        return 1.0
    raise ValueError(scheme)


class TextVectorizer:
    """เวกเตอร์ไรเซอร์ที่เขียนเองเพื่อให้ "เห็นทุกขั้น" (ต่างจาก sklearn ตรงที่เลือกสูตร TF/IDF ได้อิสระ)

    พารามิเตอร์ (ตาราง 3.67)
      tf, idf        : รูปแบบ TF และ IDF (ดู tf_weight / idf_weight)
      min_df, max_df : ตัดคำที่พบน้อยเกิน (จำนวนเอกสาร) / บ่อยเกิน (สัดส่วนเอกสาร) — คุมขนาดคลังศัพท์
      ngram_range    : (1,1) unigram · (1,2) unigram+bigram
      norm           : 'l2' (ค่าตั้งต้น — ทำให้ผลคูณจุด = โคไซน์) หรือ None
      max_features   : เก็บเฉพาะคำที่ df สูงสุด N คำ (None = ทั้งหมด)
    หลัง fit จะมี  vocab_ (list), idx_ (dict คำ→คอลัมน์), df_ (Counter), idf_ (np.array),
                   doc_len_ (ความยาวเอกสารฝึกหลังเตรียมข้อความ), avgdl_
    ข้อสังเกต: IDF และ df คำนวณจาก "เอกสารที่ fit" เท่านั้น — ในการทดลองต้อง fit ด้วยคลังรายการ
               (ซึ่งไม่ใช่ข้อมูลผู้ใช้จึงไม่ถือว่ารั่ว) แต่สถิติของผู้ใช้ทุกตัวต้องมาจากชุดฝึก (ตาราง 2.10)
    """
    def __init__(self, tf="log", idf="smooth", min_df=1, max_df=1.0, ngram_range=(1, 1),
                 stop=STOP_EN, norm="l2", max_features=None, keep_thai=False, stemmer=None):
        self.tf, self.idf, self.min_df, self.max_df = tf, idf, min_df, max_df
        self.ngram_range, self.stop, self.norm = ngram_range, stop, norm
        self.max_features, self.keep_thai, self.stemmer = max_features, keep_thai, stemmer

    def _tokens(self, docs):
        return [preprocess(d, self.stop, self.ngram_range, self.keep_thai, self.stemmer) for d in docs]

    def fit(self, docs):
        toks = self._tokens(docs)
        N = len(toks)
        df = Counter()
        for d in toks:
            df.update(set(d))
        max_df_abs = self.max_df * N if isinstance(self.max_df, float) else self.max_df
        vocab = [t for t, c in df.items() if c >= self.min_df and c <= max_df_abs]
        if self.max_features:
            vocab = sorted(vocab, key=lambda t: (-df[t], t))[: self.max_features]
        self.vocab_ = sorted(vocab)                 # เรียงตามตัวอักษรเพื่อให้ตรวจสอบด้วยมือได้ (ตาราง 3.13)
        self.idx_ = {t: k for k, t in enumerate(self.vocab_)}
        self.df_ = df
        self.N_ = N
        self.idf_ = np.array([idf_weight(df[t], N, self.idf) for t in self.vocab_])
        self.doc_len_ = np.array([len(d) for d in toks])
        self.avgdl_ = float(self.doc_len_.mean()) if N else 0.0
        self._train_tokens = toks
        return self

    def transform(self, docs, tokens=None):
        """คืน scipy.sparse CSR ขนาด (จำนวนเอกสาร × ขนาดคลังศัพท์)"""
        toks = tokens if tokens is not None else self._tokens(docs)
        rows, cols, vals = [], [], []
        for r, d in enumerate(toks):
            c = Counter(t for t in d if t in self.idx_)
            if not c:
                continue
            max_f = max(c.values())
            for t, f in c.items():
                j = self.idx_[t]
                rows.append(r); cols.append(j)
                vals.append(tf_weight(f, self.tf, max_f, len(d)) * self.idf_[j])
        X = sp.csr_matrix((vals, (rows, cols)), shape=(len(toks), len(self.vocab_)), dtype=np.float64)
        if self.norm == "l2":
            X = l2_normalize_rows(X)
        return X

    def fit_transform(self, docs):
        self.fit(docs)
        return self.transform(docs, tokens=self._train_tokens)

    def top_terms(self, vec, k=10):
        """คำที่มีน้ำหนักสูงสุด k คำของเวกเตอร์หนึ่งแถว (ใช้ "อ่าน" โปรไฟล์หรือเอกสาร — หัวข้อ 3.9.4)"""
        v = np.asarray(vec.todense()).ravel() if sp.issparse(vec) else np.asarray(vec).ravel()
        order = np.argsort(-v)[:k]
        return [(self.vocab_[j], round(float(v[j]), 4)) for j in order if v[j] > 0]


def l2_normalize_rows(X):
    """ปรับความยาวแถวเป็น 1 (L2) — เอกลักษณ์ 3.7.2: หลัง L2 แล้ว ผลคูณจุด = โคไซน์ และ ‖a−b‖² = 2(1−cos)
    แถวที่เป็นศูนย์ทั้งแถว (เอกสารไม่มีคำในคลังศัพท์) จะคงเป็นศูนย์ ไม่หารด้วยศูนย์ (ตาราง 3.68 — NaN)"""
    if sp.issparse(X):
        X = sp.csr_matrix(X, dtype=np.float64)
        n = np.sqrt(np.asarray(X.multiply(X).sum(axis=1)).ravel())
        n[n == 0] = 1.0
        return sp.diags(1.0 / n) @ X
    X = np.asarray(X, dtype=np.float64)
    n = np.linalg.norm(X, axis=1, keepdims=True)
    n[n == 0] = 1.0
    return X / n


def cosine_matrix(X, Y=None):
    """เมทริกซ์โคไซน์ทุกคู่ (X: n×d, Y: m×d) — คืน dense n×m ; ถ้า X ผ่าน L2 แล้ว = X·Yᵀ"""
    Xn = l2_normalize_rows(X)
    Yn = Xn if Y is None else l2_normalize_rows(Y)
    S = Xn @ Yn.T
    return np.asarray(S.todense()) if sp.issparse(S) else np.asarray(S)


def to_dense_row(v):
    return np.asarray(v.todense()).ravel() if sp.issparse(v) else np.asarray(v, dtype=np.float64).ravel()


# =========================================================================================
# ส่วนที่ 3  BM25 / BM25+ (หัวข้อ 3.11)
# =========================================================================================
class BM25:
    """BM25 (Robertson) พร้อมส่วนขยาย BM25+ (delta) และเลือก IDF ได้
        score(q, d) = Σ_{t∈q} idf(t) · [ f(t,d)(k1+1) / (f(t,d) + k1(1 − b + b·|d|/avgdl)) + delta ]
    k1 คุมความอิ่มตัวของความถี่ (0.5–3.0 ค่าตั้งต้น 1.5), b คุมการปรับความยาว (0–1 ค่าตั้งต้น 0.75)
    delta>0 คือ BM25+ (ป้องกันเอกสารยาวถูกลดคะแนนมากเกินควร — หัวข้อ 3.11.4)
    ตัวเลขในตาราง 3.46 ใช้ k1=1.5, b=0.75, idf แบบ bm25: ln((N−df+0.5)/(df+0.5)+1)
    """
    def __init__(self, k1=1.5, b=0.75, delta=0.0, idf="bm25"):
        self.k1, self.b, self.delta, self.idf_scheme = k1, b, delta, idf

    def fit(self, token_docs):
        self.docs = token_docs
        self.N = len(token_docs)
        self.doc_len = np.array([len(d) for d in token_docs], dtype=np.float64)
        self.avgdl = self.doc_len.mean() if self.N else 0.0
        self.df = Counter()
        for d in token_docs:
            self.df.update(set(d))
        self.tf = [Counter(d) for d in token_docs]
        self.idf = {t: idf_weight(c, self.N, self.idf_scheme) for t, c in self.df.items()}
        # เมทริกซ์ความถี่คำแบบเบาบาง (N × V) เก็บเป็น CSC เพื่อดึง "คอลัมน์ของคำ" ได้เร็ว — ทำให้ให้คะแนนข้อคำถามยาว ๆ ได้ทันที
        self.vocab_ = sorted(self.df)
        self.idx_ = {t: j for j, t in enumerate(self.vocab_)}
        rows, cols, vals = [], [], []
        for r, c in enumerate(self.tf):
            for t, f in c.items():
                rows.append(r); cols.append(self.idx_[t]); vals.append(f)
        self.TF_ = sp.csc_matrix((vals, (rows, cols)), shape=(self.N, len(self.vocab_)), dtype=np.float64)
        self.K_ = self.k1 * (1 - self.b + self.b * self.doc_len / max(self.avgdl, 1e-9))   # ตัวหารส่วนปรับความยาว ต่อเอกสาร
        return self

    def score_query(self, query_terms, term_weights=None):
        """คะแนน BM25 ของ "ทุกเอกสาร" ต่อข้อคำถามหนึ่งชุด (query_terms = รายการคำ; น้ำหนักคำใช้ได้)
        คืน np.array ยาว N — เอกสารที่ไม่มีคำในข้อคำถามเลยจะได้ 0 พอดี (ต่างจากโคไซน์ที่ได้ค่าบวกเล็ก ๆ จากคำอื่น)"""
        s = np.zeros(self.N)
        for t in query_terms:
            j = self.idx_.get(t)
            if j is None:
                continue
            w = 1.0 if term_weights is None else term_weights.get(t, 1.0)
            col = self.TF_[:, j]                                   # เฉพาะเอกสารที่มีคำ t (เบาบาง)
            docs, f = col.indices, col.data
            comp = f * (self.k1 + 1) / (f + self.K_[docs]) + self.delta
            s[docs] += w * self.idf[t] * comp
        return s

    def explain(self, query_terms, doc_index):
        """แจกแจงองค์ประกอบรายคำของเอกสารหนึ่งฉบับ (ใช้อ่านตาราง 3.46 ทีละคอลัมน์)"""
        rows = []
        K = self.k1 * (1 - self.b + self.b * self.doc_len[doc_index] / max(self.avgdl, 1e-9))
        for t in query_terms:
            f = self.tf[doc_index].get(t, 0)
            if f == 0:
                continue
            comp = f * (self.k1 + 1) / (f + K) + self.delta
            rows.append({"คำ": t, "f": f, "องค์ประกอบ TF": round(comp, 4), "idf": round(self.idf[t], 4),
                         "คะแนน": round(comp * self.idf[t], 4)})
        return pd.DataFrame(rows)


# =========================================================================================
# ส่วนที่ 4  ปริภูมิความหมาย: LSA และ Sentence Embedding (หัวข้อ 3.13)
# =========================================================================================
class LSA:
    """Latent Semantic Analysis = TruncatedSVD บนเมทริกซ์ TF-IDF แล้วปรับ L2
    k (มิติแฝง) ต้องน้อยกว่า min(จำนวนเอกสาร, ขนาดคลังศัพท์) — ในคลังเล็กอย่าง MovieMini-12 (N=12)
    ค่า k จึงเล็กมากและ "ไม่มีสถิติพอ" ให้เชื่อ (ข้อจำกัดที่บทตั้งใจให้เห็น หัวข้อ 3.13.2)"""
    def __init__(self, k=50, seed=SEED):
        self.k, self.seed = k, seed

    def fit(self, X):
        from sklearn.decomposition import TruncatedSVD
        k = int(min(self.k, min(X.shape) - 1))
        self.svd = TruncatedSVD(n_components=max(1, k), random_state=self.seed).fit(X)
        self.explained_ = self.svd.explained_variance_ratio_.sum()
        return self

    def transform(self, X):
        return l2_normalize_rows(self.svd.transform(X))

    def fit_transform(self, X):
        return self.fit(X).transform(X)


def sentence_embeddings(texts, model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=64):
    """เวกเตอร์ความหมายจาก Bi-Encoder สำเร็จรูป (ต้อง pip install sentence-transformers และมีอินเทอร์เน็ต)
    คืน np.array (n × d) ที่ปรับ L2 แล้ว ; หากโหลดไม่ได้จะคืน None ให้ผู้เรียกใช้ LSA แทน
    ข้อสังเกต: ผลลัพธ์ขึ้นกับรุ่นโมเดลและ "กลยุทธ์การรวมค่า" (mean pooling) ที่โมเดลนั้นใช้ (3.13.5)"""
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer(model_name)
        E = m.encode(list(texts), batch_size=batch_size, show_progress_bar=False, normalize_embeddings=True)
        return np.asarray(E, dtype=np.float64)
    except Exception as e:  # ไม่มีไลบรารี / ไม่มีเน็ต
        print("[คำเตือน] ใช้ sentence-transformers ไม่ได้:", str(e)[:120], "→ ใช้ LSA แทน")
        return None


# =========================================================================================
# ส่วนที่ 5  โปรไฟล์ผู้ใช้ (หัวข้อ 3.8 และ 3.12)
# =========================================================================================
def build_profile(X, hist, method="centroid", like_thr=4, dislike_thr=2, beta=0.8, gamma=0.2,
                  alpha=0.0, query_vec=None, clip_negative=True, normalize=True):
    """สร้างเวกเตอร์โปรไฟล์ในปริภูมิเดียวกับรายการ (หลักการข้อเดียวของ 3.8.1)

    X      : เมทริกซ์รายการ (n_items × d) — sparse หรือ dense ; แถวควรผ่าน L2 แล้ว
    hist   : dict {item_index: rating} ประวัติของผู้ใช้ (ดัชนีแถวของ X)
    method :
      'centroid' : ค่าเฉลี่ยของเวกเตอร์รายการที่ชอบ (rating ≥ like_thr)                     (3.8.2)
      'weighted' : Σ (r−3)·v / Σ|r−3|  ใช้ทั้งชอบและไม่ชอบ ค่ากลาง (3) ไม่มีผล               (3.8.4)
      'rocchio'  : α·q + β·mean(ชอบ) − γ·mean(ไม่ชอบ)  ค่าตั้งต้น β=0.8, γ=0.2 (กฎ β>γ)       (3.8.5)
    clip_negative=True : ตัดมิติที่ติดลบเป็น 0 ก่อนปรับ L2 (ตามตัวอย่างในหัวข้อ 3.8.6)
    คืน np.array ยาว d (ปรับ L2 แล้วเมื่อ normalize=True) — ถ้าไม่มีรายการที่ชอบเลยคืนเวกเตอร์ศูนย์
    ข้อสังเกต: เวกเตอร์ศูนย์ = "ไม่มีหลักฐาน" ต้องแยกจากคะแนนต่ำ (3.10.2) ผู้เรียกควร fallback เป็นความนิยม
    """
    d = X.shape[1]
    liked = [i for i, r in hist.items() if r >= like_thr]
    disliked = [i for i, r in hist.items() if r <= dislike_thr]

    def mean_rows(idx):
        if not idx:
            return np.zeros(d)
        M = X[idx]
        return to_dense_row(M.mean(axis=0))

    if method == "centroid":
        p = mean_rows(liked)
    elif method == "weighted":
        num = np.zeros(d); den = 0.0
        for i, r in hist.items():
            w = r - 3.0
            if w == 0:
                continue
            num += w * to_dense_row(X[i]); den += abs(w)
        p = num / den if den > 0 else num
    elif method == "rocchio":
        q = np.zeros(d) if query_vec is None else to_dense_row(query_vec)
        p = alpha * q + beta * mean_rows(liked) - gamma * mean_rows(disliked)
    else:
        raise ValueError(method)
    if clip_negative:
        p = np.clip(p, 0, None)
    if normalize:
        n = np.linalg.norm(p)
        if n > 0:
            p = p / n
    return p


class ClassifierProfile:
    """โปรไฟล์แบบ "เรียนรู้ได้" รายผู้ใช้ (หัวข้อ 3.12): Multinomial Naive Bayes หรือ Logistic Regression
    มองการแนะนำเป็นการจำแนกสองคลาส: ชอบ (rating ≥ like_thr) กับ ไม่ชอบ (≤ dislike_thr)
      * NB ต้องใช้ "จำนวนนับ" (เมทริกซ์ที่ค่าไม่ติดลบ) และมีการปรับเรียบลาปลัส alpha (นิยาม 3.11)
      * LR เรียนน้ำหนักรายมิติโดยตรง ใช้ C (ส่วนกลับของ regularization) — ต้องการข้อมูลมากกว่า (ตาราง 3.47)
    ถ้าผู้ใช้มีตัวอย่างไม่ครบทั้งสองคลาส หรือน้อยกว่า min_examples จะ "ฝึกไม่ได้" → score() คืน None
    ให้ผู้เรียกใช้โปรไฟล์ centroid แทน (บันได 3.48: ระดับ 1–2 ก่อน ระดับ 3)
    """
    def __init__(self, kind="nb", alpha=1.0, C=1.0, like_thr=4, dislike_thr=2, min_examples=2, seed=SEED):
        self.kind, self.alpha, self.C = kind, alpha, C
        self.like_thr, self.dislike_thr, self.min_examples, self.seed = like_thr, dislike_thr, min_examples, seed
        self.model = None

    def fit(self, X, hist):
        pos = [i for i, r in hist.items() if r >= self.like_thr]
        neg = [i for i, r in hist.items() if r <= self.dislike_thr]
        if len(pos) < self.min_examples or len(neg) < 1:
            self.model = None
            return self
        idx = pos + neg
        y = np.array([1] * len(pos) + [0] * len(neg))
        Xs = X[idx]
        if self.kind == "nb":
            from sklearn.naive_bayes import MultinomialNB
            self.model = MultinomialNB(alpha=self.alpha).fit(Xs, y)
        else:
            from sklearn.linear_model import LogisticRegression
            self.model = LogisticRegression(C=self.C, max_iter=1000, random_state=self.seed).fit(Xs, y)
        return self

    def score(self, X):
        """ความน่าจะเป็น P(ชอบ | เนื้อหา) ของทุกรายการ หรือ None เมื่อฝึกไม่ได้"""
        if self.model is None:
            return None
        return self.model.predict_proba(X)[:, 1]


# =========================================================================================
# ส่วนที่ 6  ฟังก์ชันจับคู่ การจัดอันดับ Top-K และ MMR (หัวข้อ 3.9–3.10)
# =========================================================================================
def match_scores(X, profile):
    """คะแนนจับคู่ = ผลคูณจุดระหว่างทุกแถวของ X กับโปรไฟล์ (เมื่อทั้งคู่ผ่าน L2 = โคไซน์) คืน np.array ยาว n_items"""
    s = X @ profile
    return np.asarray(s).ravel()


def item_knn_scores(S, liked_idx, agg="mean", k=None):
    """Item-kNN เชิงเนื้อหา (3.9.5): เทียบผู้สมัครกับรายการที่ชอบ "ทีละรายการ" แล้วรวม
    S: เมทริกซ์ความคล้าย n×n ; agg='mean' เฉลี่ย, 'max' เอาค่าสูงสุด ; k = ใช้เฉพาะ k เพื่อนบ้านที่คล้ายที่สุด"""
    if not liked_idx:
        return np.zeros(S.shape[0])
    sims = S[:, liked_idx]                       # n × |liked|
    if k is not None and k < sims.shape[1]:
        sims = -np.sort(-sims, axis=1)[:, :k]
    return sims.mean(axis=1) if agg == "mean" else sims.max(axis=1)


def explain_terms(vectorizer, profile, item_vec, k=8):
    """อธิบายผล (3.9.4): คำใดทำให้คะแนนสูง = ส่วนร่วมรายมิติ p_j · v_j เรียงจากมากไปน้อย"""
    v = to_dense_row(item_vec)
    contrib = profile * v
    order = np.argsort(-contrib)[:k]
    return [(vectorizer.vocab_[j], round(float(profile[j]), 4), round(float(v[j]), 4), round(float(contrib[j]), 4))
            for j in order if contrib[j] > 0]


def top_k(scores, seen, K=10, min_score=None, item_ids=None, candidates=None):
    """ห้าขั้นของ 3.10.1: รวบรวมผู้สมัคร → (คะแนนมีแล้ว) → กรองที่เคยเห็น → เรียง+ตัดสินเสมอ → เกณฑ์ขั้นต่ำ → ตัด K
    seen       : เซตดัชนีรายการที่ผู้ใช้เคยมีปฏิสัมพันธ์ในชุดฝึก (ห้ามแนะนำซ้ำ)
    candidates : เซตดัชนีผู้สมัคร (None = ทั้งคลัง = full-catalog ranking นิยาม 2.7)
    ตัดสินเสมอ : คะแนนเท่ากันเรียงตามดัชนี/รหัสรายการ (กฎที่ทำซ้ำได้ของบท — 3.10.2) ผ่าน np.lexsort
    คืน list ดัชนี (ยาว ≤ K)"""
    n = len(scores)
    cand = np.arange(n) if candidates is None else np.asarray(sorted(candidates))
    if seen:
        cand = cand[~np.isin(cand, list(seen))]
    if min_score is not None:
        cand = cand[scores[cand] >= min_score]
    if len(cand) == 0:
        return []
    key2 = cand if item_ids is None else np.asarray([item_ids[i] for i in cand])
    order = np.lexsort((key2, -scores[cand]))     # เรียงคะแนนมาก→น้อย แล้วรหัสน้อย→มากเมื่อเสมอ
    return [int(i) for i in cand[order][:K]]


def mmr_rerank(scores, S, K=10, lam=0.7, seen=None, pool=None, min_score=None, normalize=True):
    """Maximal Marginal Relevance (นิยาม 3.8): เลือกทีละรายการแบบละโมบ
        argmax_i  λ·rel(i) − (1−λ)·max_{j∈เลือกแล้ว} sim(i, j)
    λ=1 → เรียงตามความเกี่ยวข้องล้วน ; λ→0 → เน้นความหลากหลาย ; ค่านิยม 0.7 (ตัวอย่าง 3.9)
    pool: จำนวนผู้สมัครต้นทางที่นำมาจัดอันดับซ้ำ (ค่าตั้งต้น = 5K) เพื่อประหยัดเวลาในคลังใหญ่
    normalize=True จะปรับ rel ด้วย min-max ให้อยู่ในมาตรา 0–1 เดียวกับ sim (จำเป็นเมื่อคะแนนเป็น BM25/RRF)
    normalize=False ใช้คะแนนดิบ (ตรงกับตัวอย่างการคำนวณ 3.9 ในตำรา ซึ่ง rel อยู่ในช่วง 0–1 อยู่แล้ว)"""
    pool = pool or 5 * K
    base = top_k(scores, seen or set(), K=pool, min_score=min_score)
    if not base:
        return []
    rel = scores[base]
    if normalize and rel.max() > rel.min():
        rel = (rel - rel.min()) / (rel.max() - rel.min())
    chosen, remaining = [], list(range(len(base)))
    while remaining and len(chosen) < K:
        best, best_val = None, -1e18
        for r in remaining:
            div = max(S[base[r], base[c]] for c in chosen) if chosen else 0.0
            val = lam * rel[r] - (1 - lam) * div
            if val > best_val:
                best, best_val = r, val
        chosen.append(best); remaining.remove(best)
    return [base[c] for c in chosen]


# =========================================================================================
# ส่วนที่ 7  ผสานคะแนน · ปรับเทียบ (หัวข้อ 3.18)
# =========================================================================================
def minmax(x):
    """ปรับมาตรา 0–1 (ข้อบังคับก่อนผสมแบบนูน — ภาพที่ 3.32) ; ถ้าค่าคงที่คืนศูนย์ทั้งหมด"""
    x = np.asarray(x, dtype=np.float64)
    lo, hi = np.nanmin(x), np.nanmax(x)
    return (x - lo) / (hi - lo) if hi > lo else np.zeros_like(x)


def zscore(x):
    x = np.asarray(x, dtype=np.float64)
    s = x.std()
    return (x - x.mean()) / s if s > 0 else np.zeros_like(x)


def convex_fusion(score_list, weights=None, scale="minmax"):
    """ผสมแบบนูน: Σ w_k · scale(s_k) โดย Σw = 1 (ตัวอย่าง 3.22) — ผู้ชนะเปลี่ยนตาม w จึงต้องปรับ w บนชุดตรวจสอบ"""
    m = len(score_list)
    w = np.ones(m) / m if weights is None else np.asarray(weights, dtype=np.float64)
    w = w / w.sum()
    f = minmax if scale == "minmax" else zscore
    return sum(wi * f(s) for wi, s in zip(w, score_list))


def rrf_fusion(score_list, k=60, weights=None):
    """Reciprocal Rank Fusion: Σ 1/(k + rank_i) ใช้เฉพาะ "อันดับ" จึงไม่ต้องปรับมาตรา (k=60 ตามธรรมเนียม)"""
    n = len(score_list[0])
    out = np.zeros(n)
    w = np.ones(len(score_list)) if weights is None else np.asarray(weights)
    for wi, s in zip(w, score_list):
        ranks = np.empty(n)
        ranks[np.argsort(-np.asarray(s), kind="stable")] = np.arange(1, n + 1)
        out += wi / (k + ranks)
    return out


class PlattCalibrator:
    """Platt scaling (นิยาม 3.19): p = σ(a·s + b) ประมาณ a, b จากชุดตรวจสอบด้วย log-loss ต่ำสุด
    (ใช้ LogisticRegression 1 มิติ) — ห้ามใช้ชุดทดสอบปรับเทียบ (รั่วไหลแบบที่ 5 ในตาราง 2.10)"""
    def fit(self, scores, labels):
        from sklearn.linear_model import LogisticRegression
        self.lr = LogisticRegression(C=1e6, max_iter=1000).fit(np.asarray(scores).reshape(-1, 1), labels)
        self.a_, self.b_ = float(self.lr.coef_[0, 0]), float(self.lr.intercept_[0])
        return self
    def predict_proba(self, scores):
        return self.lr.predict_proba(np.asarray(scores).reshape(-1, 1))[:, 1]


def expected_calibration_error(probs, labels, n_bins=10):
    """ECE (นิยาม 3.20): แบ่งถังตามความมั่นใจ แล้วเฉลี่ย |สัดส่วนถูกจริง − ความมั่นใจเฉลี่ย| ถ่วงด้วยขนาดถัง
    คืน (ece, ตารางรายถัง) — ตัวอย่าง 3.23 ใช้ 5 ถัง"""
    probs, labels = np.asarray(probs), np.asarray(labels)
    edges = np.linspace(0, 1, n_bins + 1)
    rows, ece = [], 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (probs > lo) & (probs <= hi) if lo > 0 else (probs >= lo) & (probs <= hi)
        if m.sum() == 0:
            continue
        acc, conf = labels[m].mean(), probs[m].mean()
        ece += m.sum() / len(probs) * abs(acc - conf)
        rows.append({"ถัง": f"({lo:.1f}, {hi:.1f}]", "จำนวน": int(m.sum()), "สัดส่วนถูกจริง": round(acc, 4),
                     "ความมั่นใจเฉลี่ย": round(conf, 4), "|ผลต่าง|": round(abs(acc - conf), 4)})
    return ece, pd.DataFrame(rows)


# =========================================================================================
# ส่วนที่ 8  โมเดลทำนายคะแนน (Rating Prediction) — เพื่อวัด MAE/RMSE ของบทที่ 2 (2.1)
# =========================================================================================
class ContentKNNRegressor:
    """ทำนายคะแนน r̂(u,i) จากรายการที่ u เคยให้คะแนน ถ่วงด้วยความคล้าย "เชิงเนื้อหา" ระหว่างรายการ
        r̂(u,i) = μ_u + Σ_j sim(i,j)·(r_uj − μ_u) / Σ_j |sim(i,j)|     (j = k เพื่อนบ้านที่คล้ายที่สุดที่ u เคยให้คะแนน)
    * ไม่ต้องใช้ผู้ใช้คนอื่นเลย (ข้อดีข้อแรกของ content-based 3.19.1)
    * ถ้าไม่มีเพื่อนบ้าน (sim=0 ทุกคู่) จะถอยไปใช้ค่าเฉลี่ยของผู้ใช้ μ_u ; ถ้าผู้ใช้ใหม่ใช้ค่าเฉลี่ยรวม μ
    * shrink: ลดอิทธิพลเมื่อผลรวมความคล้ายเล็ก (กัน overfit เพื่อนบ้านน้อย)
    """
    def __init__(self, X, k=20, shrink=0.0, min_sim=0.0):
        self.Xn = l2_normalize_rows(X)
        self.k, self.shrink, self.min_sim = k, shrink, min_sim

    def fit(self, train):
        """train: DataFrame มีคอลัมน์ u (ดัชนีผู้ใช้), i (ดัชนีรายการ), r (คะแนน) — ต้องเป็น "ชุดฝึกเท่านั้น" """
        self.mu = float(train["r"].mean())
        self.user_hist = {u: (g["i"].to_numpy(), g["r"].to_numpy()) for u, g in train.groupby("u")}
        self.user_mean = {u: float(r.mean()) for u, (_, r) in self.user_hist.items()}
        return self

    def predict_one(self, u, i):
        if u not in self.user_hist:
            return self.mu
        items, ratings = self.user_hist[u]
        mu_u = self.user_mean[u]
        sims = np.asarray((self.Xn[items] @ self.Xn[i].T).todense()).ravel() if sp.issparse(self.Xn) \
            else self.Xn[items] @ self.Xn[i]
        m = sims > self.min_sim
        if m.sum() == 0:
            return mu_u
        sims, ratings = sims[m], ratings[m]
        if self.k and len(sims) > self.k:
            top = np.argsort(-sims)[: self.k]
            sims, ratings = sims[top], ratings[top]
        den = np.abs(sims).sum() + self.shrink
        return mu_u + float((sims * (ratings - mu_u)).sum() / den) if den > 0 else mu_u

    def predict(self, pairs):
        """pairs: DataFrame คอลัมน์ u, i → np.array คะแนนทำนาย (ผู้เรียกควร clip ให้อยู่ในช่วงคะแนนเอง)
        คำนวณเป็นกลุ่มต่อผู้ใช้ (เมทริกซ์ความคล้าย |ประวัติ| × |รายการที่ต้องทำนาย|) เร็วกว่าเรียก predict_one ทีละคู่มาก"""
        out = np.full(len(pairs), self.mu)
        pu, pi = pairs["u"].to_numpy(), pairs["i"].to_numpy()
        for u in np.unique(pu):
            m = np.where(pu == u)[0]
            if u not in self.user_hist:
                continue
            items, ratings = self.user_hist[u]; mu_u = self.user_mean[u]
            S = self.Xn[items] @ self.Xn[pi[m]].T
            S = np.asarray(S.todense()) if sp.issparse(S) else np.asarray(S)      # |hist| × |targets|
            S[S <= self.min_sim] = 0.0
            if self.k and S.shape[0] > self.k:                                   # เก็บเฉพาะ k เพื่อนบ้านที่คล้ายที่สุดต่อคอลัมน์
                thr = -np.partition(-S, self.k - 1, axis=0)[self.k - 1]
                S = np.where(S >= thr, S, 0.0)
            den = np.abs(S).sum(axis=0) + self.shrink
            num = (S * (ratings - mu_u)[:, None]).sum(axis=0)
            pred = np.where(den > 0, mu_u + num / np.where(den > 0, den, 1.0), mu_u)
            pred = np.where(np.abs(S).sum(axis=0) > 0, pred, mu_u)
            out[m] = pred
        return out


class MeanRatingBaseline:
    """ระบบฐานสำหรับ RMSE/MAE: 'global' = ค่าเฉลี่ยรวม, 'user' = ค่าเฉลี่ยผู้ใช้, 'item' = ค่าเฉลี่ยรายการ,
    'user_item' = μ + b_u + b_i (baseline ของ Koren) — ระบบใหม่ที่แพ้ค่าเฉลี่ยผู้ใช้คือสัญญาณเตือน"""
    def __init__(self, kind="user"):
        self.kind = kind
    def fit(self, train):
        self.mu = float(train["r"].mean())
        self.bu = (train.groupby("u")["r"].mean() - self.mu).to_dict()
        self.bi = (train.groupby("i")["r"].mean() - self.mu).to_dict()
        return self
    def predict(self, pairs):
        out = []
        for u, i in zip(pairs["u"], pairs["i"]):
            if self.kind == "global":
                out.append(self.mu)
            elif self.kind == "user":
                out.append(self.mu + self.bu.get(u, 0.0))
            elif self.kind == "item":
                out.append(self.mu + self.bi.get(i, 0.0))
            else:
                out.append(self.mu + self.bu.get(u, 0.0) + self.bi.get(i, 0.0))
        return np.array(out)


# =========================================================================================
# ส่วนที่ 9  ตัวชี้วัดของบทที่ 2
# =========================================================================================
# ---- 2.1 ตัวชี้วัดการทำนายคะแนน ----
def mae(y_true, y_pred):
    """Mean Absolute Error (2.1.1) — ทุกความผิดพลาดหนักเท่ากัน"""
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true, y_pred):
    """Root Mean Squared Error (2.1.2) — ลงโทษความผิดพลาดใหญ่มากกว่า (ตัวอย่าง 2.2: MAE เท่ากันแต่ RMSE ต่างกัน)"""
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


# ---- 2.2–2.3 ตัวชี้วัดเชิงอันดับ "รายผู้ใช้" (rec_list = ลำดับรายการที่แนะนำ, rel = เซตรายการที่เกี่ยวข้อง) ----
def precision_at_k(rec, rel, K):
    """P@K = |Top-K ∩ Rel| / K  (2.2.1) ตัวหารคือ K เสมอ แม้ผู้ใช้จะมี Rel น้อยกว่า K"""
    return len(set(rec[:K]) & rel) / K


def recall_at_k(rec, rel, K):
    """R@K = |Top-K ∩ Rel| / |Rel|  (2.2.2) ; ผู้ใช้ที่ไม่มี Rel เลยควรถูกตัดออกก่อนประเมิน"""
    return len(set(rec[:K]) & rel) / len(rel) if rel else 0.0


def f1_at_k(rec, rel, K):
    p, r = precision_at_k(rec, rel, K), recall_at_k(rec, rel, K)
    return 2 * p * r / (p + r) if p + r > 0 else 0.0


def hit_at_k(rec, rel, K):
    """HR@K = 1 ถ้ามีรายการเกี่ยวข้องอย่างน้อยหนึ่งใน Top-K (2.2.4) — นิยมกับ leave-one-out"""
    return 1.0 if set(rec[:K]) & rel else 0.0


def rr_at_k(rec, rel, K):
    """Reciprocal Rank (2.3.1): 1/อันดับของรายการเกี่ยวข้อง "ตัวแรก" ; 0 ถ้าไม่มีใน K"""
    for pos, it in enumerate(rec[:K], start=1):
        if it in rel:
            return 1.0 / pos
    return 0.0


def ap_at_k(rec, rel, K):
    """Average Precision@K (2.3.2): เฉลี่ย P@i เฉพาะตำแหน่ง i ที่เป็นรายการเกี่ยวข้อง หารด้วย min(|Rel|, K)"""
    if not rel:
        return 0.0
    hits, s = 0, 0.0
    for pos, it in enumerate(rec[:K], start=1):
        if it in rel:
            hits += 1
            s += hits / pos
    return s / min(len(rel), K)


def dcg_at_k(gains):
    """DCG = Σ gain_i / log2(i+1) (2.3.3) — ตำแหน่ง 1 หารด้วย log2(2)=1, ตำแหน่ง 2 หารด้วย log2(3)…"""
    return float(sum(g / math.log2(i + 1) for i, g in enumerate(gains, start=1)))


def ndcg_at_k(rec, rel, K, rel_grades=None, gain_type="linear"):
    """nDCG@K = DCG@K / IDCG@K
    rel_grades : dict item→ระดับความเกี่ยวข้อง (เช่น rating) ถ้า None ใช้ 1/0 (binary)
    gain_type  : 'linear' gain=grade ; 'exp' gain=2^grade−1 (ต้องประกาศแบบที่ใช้ — ตาราง 2.16 ขั้น 4)"""
    def g(it):
        grade = (rel_grades.get(it, 0.0) if rel_grades else (1.0 if it in rel else 0.0))
        if it not in rel:
            grade = 0.0
        return (2 ** grade - 1) if gain_type == "exp" else grade
    gains = [g(it) for it in rec[:K]]
    ideal = sorted([g(it) for it in rel], reverse=True)[:K]
    idcg = dcg_at_k(ideal)
    return dcg_at_k(gains) / idcg if idcg > 0 else 0.0


def auc_user(scores, pos_idx, neg_idx):
    """AUC (นิยาม 2.1) นับคู่ทีละคู่: สัดส่วนคู่ (บวก, ลบ) ที่รายการบวกได้คะแนนสูงกว่า (เสมอนับ 0.5)"""
    if len(pos_idx) == 0 or len(neg_idx) == 0:
        return np.nan
    ps, ns = scores[list(pos_idx)], scores[list(neg_idx)]
    diff = ps[:, None] - ns[None, :]
    return float(((diff > 0).sum() + 0.5 * (diff == 0).sum()) / diff.size)


# ---- 2.5 ตัวชี้วัด Beyond-Accuracy (ระดับระบบ / ระดับรายการแนะนำ) ----
def catalog_coverage(rec_lists, n_items):
    """Coverage@K (2.5.1) = |⋃_u Top-K(u)| / |I| — รายการนับครั้งเดียวไม่ว่าจะโผล่กี่คน"""
    s = set()
    for r in rec_lists:
        s.update(r)
    return len(s) / n_items


def gini_exposure(rec_lists, n_items):
    """Gini ของการเปิดเผย (2.5.2): x_i = จำนวนครั้งที่รายการ i ถูกแนะนำ "เรียงจากน้อยไปมาก" (ขั้นบังคับ)
        G = Σ (2i − n − 1) x_i / (n Σ x_i)   ; 0 = เท่าเทียม, →1 = กระจุกที่รายการเดียว
    รายการที่ไม่เคยถูกแนะนำนับเป็น x=0 ด้วย (n = ทั้งคลัง)"""
    x = np.zeros(n_items)
    for r in rec_lists:
        for it in r:
            x[it] += 1
    x = np.sort(x)
    n = len(x)
    if x.sum() == 0:
        return 0.0
    i = np.arange(1, n + 1)
    return float(((2 * i - n - 1) * x).sum() / (n * x.sum()))


def head_items_from_train(train_item_counts, n_items, share=1 / 3):
    """เส้นแบ่งหัว–หาง (นิยาม 2.2, ตัวอย่าง 2.8): เรียงตามความนิยม "จากชุดฝึก" แล้วสะสมจนถึง share ของปฏิสัมพันธ์ทั้งหมด
    คืน set ของ head items (ที่เหลือ = long tail) — ต้องประกาศเกณฑ์ที่ใช้ทุกครั้ง และห้ามคำนวณจากข้อมูลทั้งหมด (รั่ว)"""
    counts = np.zeros(n_items)
    for i, c in train_item_counts.items():
        counts[i] = c
    order = np.argsort(-counts, kind="stable")
    cum, total, head = 0.0, counts.sum(), set()
    for i in order:
        if counts[i] == 0:
            break
        head.add(int(i)); cum += counts[i]
        if cum >= share * total:
            break
    return head


def long_tail_ratio(rec, head, K):
    """LongTailRatio@K = สัดส่วนรายการใน Top-K ที่ "ไม่ใช่" head"""
    top = rec[:K]
    return sum(1 for it in top if it not in head) / K if top else 0.0


def intra_list_diversity(rec, F, K):
    """ILD@K (นิยาม 2.3) = 2/(K(K−1)) Σ_{i<j} (1 − cos(f_i, f_j)) วัดจากคุณลักษณะรายการ F (เช่น เวกเตอร์แนว)
    ค่าเข้าใกล้ 1 = ต่างกันมาก ; รายการที่มีชิ้นเดียวคืน 0"""
    top = rec[:K]
    if len(top) < 2:
        return 0.0
    M = l2_normalize_rows(F[top])
    S = M @ M.T
    S = np.asarray(S.todense()) if sp.issparse(S) else np.asarray(S)
    n = len(top)
    iu = np.triu_indices(n, k=1)
    return float((1 - S[iu]).mean())


def novelty(rec, p_item, K):
    """Novelty@K (2.5.4) = (1/K) Σ −log2 p(i) ; p(i) = "สัดส่วนผู้ใช้" ในชุดฝึกที่เคยมีปฏิสัมพันธ์กับ i (หน่วยบิต)
    รายการที่ไม่มีใครเคยเห็นเลย p=0 → ใช้ p = 1/(จำนวนผู้ใช้+1) กัน log(0) (ต้องประกาศ)"""
    top = rec[:K]
    if not top:
        return 0.0
    return float(np.mean([-math.log2(max(p_item[it], p_item.min_p)) for it in top]))


class ItemPopularity:
    """เก็บ p(i) = สัดส่วนผู้ใช้ที่เคยมีปฏิสัมพันธ์กับ i (คำนวณจากชุดฝึกเท่านั้น) และจำนวนปฏิสัมพันธ์รายรายการ"""
    def __init__(self, train, n_items, n_users):
        c_users = train.groupby("i")["u"].nunique()
        self.p = np.zeros(n_items)
        self.p[c_users.index.to_numpy()] = c_users.to_numpy() / n_users
        self.count = train.groupby("i").size().to_dict()
        self.min_p = 1.0 / (n_users + 1)
    def __getitem__(self, i):
        return self.p[i]


def calibration_error(hist_items, rec, G, K):
    """Calibration (2.5.5): 0.5 Σ_c |q(c) − p(c)| ; p = สัดส่วนแนวในประวัติ, q = สัดส่วนแนวในรายการแนะนำ
    G = เมทริกซ์รายการ×แนว (0/1) ; ค่าใกล้ 0 = สอดคล้องกับประวัติ"""
    def dist(idx):
        if len(idx) == 0:
            return None
        v = np.asarray(G[list(idx)].sum(axis=0)).ravel().astype(float)
        return v / v.sum() if v.sum() > 0 else v
    p, q = dist(hist_items), dist(rec[:K])
    if p is None or q is None:
        return np.nan
    return float(0.5 * np.abs(q - p).sum())


def parity_gap(per_user_df, metric, group_col):
    """Fairness ฝั่งผู้ใช้ (2.5.6): ผลต่างของ Primary Metric ระหว่างกลุ่มผู้ใช้ (เช่น ประวัติยาว vs สั้น)"""
    g = per_user_df.groupby(group_col)[metric].mean()
    return float(g.max() - g.min()), g


# ---- ตัวประเมินหลัก: จัดอันดับ + คำนวณทุกตัวชี้วัดในรอบเดียว ----
def evaluate_ranking(score_fn, users, train_hist, test_rel, n_items, K=10, item_feat=None, pop=None,
                     head=None, genre_mat=None, rel_grades=None, candidate="full", m_neg=99, seed=SEED,
                     rerank_fn=None, min_score=None, item_ids=None, pool=None):
    """ประเมินโมเดลจัดอันดับหนึ่งตัวบนผู้ใช้ทุกคน คืน (ตารางรายผู้ใช้, สรุปค่าเฉลี่ย, dict รายการแนะนำ)

    score_fn(u)  → np.array คะแนนทุกรายการ (ดัชนี 0..n_items-1) ของผู้ใช้ u ; คืน None = ไม่มีหลักฐาน
    train_hist   : dict u → {item_index: rating} จากชุดฝึก (ใช้กรองที่เคยเห็น และ calibration)
    test_rel     : dict u → set(item_index) ที่เกี่ยวข้องในชุดที่กันไว้ (เช่น rating ≥ 4)
    candidate    : 'full' = จัดอันดับทั้งคลังที่ยังไม่เห็น (นิยาม 2.7) ; 'sampled' = เป้าหมาย + m_neg รายการลบสุ่ม
                   (ต้องรายงานว่าเป็น sampled metric และห้ามเทียบข้าม m — 2.9.5) ; รายการลบไม่ถูกวางไว้ตำแหน่งแรก
    rerank_fn(u, scores) → list ดัชนี (เช่น MMR) ใช้แทนการเรียงตามคะแนนเมื่อกำหนด
    item_feat    : เมทริกซ์คุณลักษณะสำหรับ ILD (เช่น เวกเตอร์แนว) ; genre_mat สำหรับ calibration
    pop          : ItemPopularity สำหรับ Novelty ; head : เซต head items สำหรับ LongTail
    pool         : เซตดัชนีรายการที่ "ให้บริการได้" (servable catalog เช่น รายการที่อยู่ใน k-core ของชุดฝึก)
                   None = ทั้งคลัง ; เมื่อกำหนด ทั้ง candidate, AUC และตัวหารของ Coverage/Gini จะใช้เซตนี้ (ต้องประกาศ)
    ผู้ใช้ที่ไม่มีรายการเกี่ยวข้องในชุดทดสอบจะถูกข้าม (ประกาศจำนวนที่ประเมินจริงเสมอ)
    """
    rng = np.random.default_rng(seed)
    rows, rec_lists, recs = [], [], {}
    pool_arr = np.arange(n_items) if pool is None else np.asarray(sorted(pool))
    pool_set = None if pool is None else set(int(i) for i in pool_arr)
    n_cat = len(pool_arr)
    for u in users:
        rel = test_rel.get(u, set())
        if not rel:
            continue
        seen = set(train_hist.get(u, {}).keys())
        s = score_fn(u)
        if s is None:
            s = np.zeros(n_items)
        s = np.asarray(s, dtype=np.float64)
        cands = pool_set
        if candidate == "sampled":
            unseen = np.setdiff1d(pool_arr, list(seen | rel))
            neg = rng.choice(unseen, size=min(m_neg, len(unseen)), replace=False)
            cands = set(neg.tolist()) | rel
        if rerank_fn is not None:
            rec = rerank_fn(u, s)
        else:
            rec = top_k(s, seen, K, min_score=min_score, item_ids=item_ids, candidates=cands)
        recs[u] = rec
        rec_lists.append(rec)
        row = {"user": u, "n_rel": len(rel), "n_hist": len(seen),
               "P@K": precision_at_k(rec, rel, K), "R@K": recall_at_k(rec, rel, K), "F1@K": f1_at_k(rec, rel, K),
               "HR@K": hit_at_k(rec, rel, K), "RR@K": rr_at_k(rec, rel, K), "AP@K": ap_at_k(rec, rel, K),
               "nDCG@K": ndcg_at_k(rec, rel, K, rel_grades.get(u) if rel_grades else None)}
        if candidate == "full":
            neg_pool = np.setdiff1d(pool_arr, list(seen | rel))
            if len(neg_pool) > 500:
                neg_pool = rng.choice(neg_pool, 500, replace=False)   # ประมาณ AUC ด้วยตัวอย่างลบ 500 ชิ้น
            row["AUC"] = auc_user(s, list(rel), neg_pool)
        if item_feat is not None:
            row["ILD@K"] = intra_list_diversity(rec, item_feat, K)
        if pop is not None:
            row["Novelty@K"] = novelty(rec, pop, K)
        if head is not None:
            row["LongTail@K"] = long_tail_ratio(rec, head, K)
        if genre_mat is not None:
            row["CalibErr"] = calibration_error(list(seen), rec, genre_mat, K)
        rows.append(row)
    per_user = pd.DataFrame(rows)
    summary = per_user.drop(columns=["user", "n_rel", "n_hist"]).mean(numeric_only=True).to_dict()
    summary["MRR@K"] = summary.pop("RR@K", 0.0)
    summary["MAP@K"] = summary.pop("AP@K", 0.0)
    summary["Coverage@K"] = catalog_coverage(rec_lists, n_cat)      # ตัวหาร = ขนาดคลังที่ให้บริการได้ (pool) หรือทั้งคลัง
    summary["Gini"] = gini_exposure(rec_lists, n_items) if pool is None else gini_exposure(
        [[int(np.searchsorted(pool_arr, it)) for it in r if it in pool_set] for r in rec_lists], n_cat)
    summary["n_users"] = len(per_user)
    return per_user, summary, recs


# =========================================================================================
# ส่วนที่ 10  การออกแบบการทดลองออฟไลน์ (หัวข้อ 2.7–2.10, 2.14)
# =========================================================================================
def encode_ids(df, user_col="userId", item_col="movieId", items_df=None, item_id_col="movieId"):
    """แปลงรหัสผู้ใช้/รายการเป็นดัชนี 0..n-1 (คอลัมน์ u, i) — รายการอ้างอิงตาม items_df เพื่อให้คลังครบทุกชิ้น
    แม้ไม่มีใครให้คะแนน (จำเป็นต่อ Coverage ที่ตัวหารคือ "ทั้งคลัง")"""
    users = sorted(df[user_col].unique())
    u_map = {u: k for k, u in enumerate(users)}
    if items_df is not None:
        items = list(items_df[item_id_col])
    else:
        items = sorted(df[item_col].unique())
    i_map = {it: k for k, it in enumerate(items)}
    out = df.copy()
    out["u"] = out[user_col].map(u_map)
    out["i"] = out[item_col].map(i_map)
    out = out.dropna(subset=["u", "i"]).astype({"u": int, "i": int})
    if "rating" in out.columns:
        out["r"] = out["rating"].astype(float)
    return out, u_map, i_map


def kcore_filter(df, k_user=5, k_item=5, max_iter=20):
    """กรอง k-core "ก่อน" แบ่งข้อมูล (รูปแบบรั่วไหลที่ 3 ในตาราง 2.10 คือกรองหลังแบ่ง) — วนจนนิ่ง"""
    d = df
    for _ in range(max_iter):
        n0 = len(d)
        uc = d.groupby("u").size(); ic = d.groupby("i").size()
        d = d[d["u"].isin(uc[uc >= k_user].index) & d["i"].isin(ic[ic >= k_item].index)]
        if len(d) == n0:
            break
    return d


def split_random(df, test_frac=0.2, seed=SEED):
    """แบบที่ 1 สุ่มโดยไม่สนเวลา — ง่ายแต่ "ใช้อนาคตทำนายอดีต" เมื่อข้อมูลมีเวลากำกับ (2.8.1)"""
    rng = np.random.default_rng(seed)
    m = rng.random(len(df)) < test_frac
    return df[~m].copy(), df[m].copy()


def split_global_temporal(df, test_frac=0.2, time_col="timestamp"):
    """แบบที่ 2 จุดตัดเวลาจุดเดียวทั้งระบบ: ก่อน = ฝึก หลัง = ทดสอบ (จำลองการใช้งานจริงตรงที่สุด)
    ข้อสังเกต: ผู้ใช้บางคนอาจไม่มีข้อมูลฝึกเลย (ผู้ใช้ใหม่จริง ๆ) — ต้องตัดสินใจว่าจะประเมินหรือไม่และประกาศ"""
    cut = df[time_col].quantile(1 - test_frac)
    return df[df[time_col] <= cut].copy(), df[df[time_col] > cut].copy()


def split_per_user_temporal(df, test_frac=0.2, time_col="timestamp", min_train=1):
    """แบบที่ 3 กันส่วนท้าย test_frac ของ "แต่ละคน" — ทุกคนมีชุดทดสอบ แต่จุดตัดไม่ตรงกัน (ยังรั่วข้ามผู้ใช้ได้)"""
    tr, te = [], []
    for u, g in df.sort_values(time_col).groupby("u"):
        n_te = int(np.ceil(len(g) * test_frac))
        n_te = min(n_te, max(0, len(g) - min_train))
        if n_te == 0:
            tr.append(g); continue
        tr.append(g.iloc[:-n_te]); te.append(g.iloc[-n_te:])
    return pd.concat(tr), (pd.concat(te) if te else df.iloc[0:0].copy())


def split_leave_one_out(df, time_col="timestamp", only_relevant=False, like_thr=4, seed=SEED):
    """แบบที่ 4 กันเหตุการณ์ "ล่าสุด" ของแต่ละคนไว้หนึ่งชิ้น (ไม่มีเวลา → สุ่มหนึ่งชิ้น)
    only_relevant=True: กันเฉพาะรายการที่ชอบ (rating ≥ like_thr) เพื่อให้ทุกผู้ใช้มีเป้าหมายวัด HR/MRR ได้
    ความแปรปรวนสูงมากเพราะตัดสินผู้ใช้ด้วยเหตุการณ์เดียว (2.8.1)"""
    rng = np.random.default_rng(seed)
    tr, te = [], []
    for u, g in df.groupby("u"):
        pool = g[g["r"] >= like_thr] if only_relevant and "r" in g else g
        if len(g) < 2 or len(pool) == 0:
            tr.append(g); continue
        if time_col in g.columns:
            hold = pool.sort_values(time_col).index[-1]
        else:
            hold = rng.choice(pool.index)
        te.append(g.loc[[hold]]); tr.append(g.drop(index=hold))
    return pd.concat(tr), pd.concat(te)


def three_way_split(df, protocol="global_temporal", valid_frac=0.1, test_frac=0.2, seed=SEED, **kw):
    """แบ่งสามชุด Train → Validation → Test (ตาราง 2.10 แถวที่ 5: เลือก hyperparameter บน validation เท่านั้น
    และแตะ test ครั้งเดียวตอนท้าย) โดยใช้โปรโตคอลเดียวกันสองครั้ง: ตัด test ออกก่อน แล้วตัด valid จากที่เหลือ"""
    f = {"random": split_random, "global_temporal": split_global_temporal,
         "per_user_temporal": split_per_user_temporal}[protocol]
    if protocol == "random":
        rest, test = f(df, test_frac, seed=seed)
        train, valid = f(rest, valid_frac / (1 - test_frac), seed=seed + 1)
    else:
        rest, test = f(df, test_frac, **kw)
        train, valid = f(rest, valid_frac / (1 - test_frac), **kw)
    return train, valid, test


def leakage_check(train, test, time_col="timestamp"):
    """ตรวจรั่วไหลเชิงเวลา (2.8.4): นับเหตุการณ์ในชุดฝึกที่เกิด "หลัง" เหตุการณ์แรกของชุดทดสอบ
    0 = ปลอดภัยแบบ global temporal ; >0 = มีการรั่วข้ามเวลา (ยอมรับได้เฉพาะเมื่อประกาศ เช่น per-user split)"""
    if time_col not in train.columns or len(test) == 0:
        return {"future_events_in_train": None}
    t0 = test[time_col].min()
    n_future = int((train[time_col] > t0).sum())
    return {"future_events_in_train": n_future, "share": n_future / max(1, len(train)),
            "train_max_time": pd.to_datetime(train[time_col].max(), unit="s"),
            "test_min_time": pd.to_datetime(t0, unit="s")}


def hist_from(df):
    """dict u → {i: r} จาก DataFrame (ใช้กับชุดฝึก)"""
    return {u: dict(zip(g["i"], g["r"])) for u, g in df.groupby("u")}


def relevant_from(df, like_thr=4.0):
    """dict u → set(i) ที่ rating ≥ like_thr (นิยามความเกี่ยวข้อง — ต้องประกาศ ขั้นที่ 4 ของโปรโตคอล)
    และ dict u → {i: r} สำหรับ nDCG แบบ graded"""
    rel, grades = {}, {}
    for u, g in df.groupby("u"):
        m = g["r"] >= like_thr
        rel[u] = set(g.loc[m, "i"])
        grades[u] = dict(zip(g.loc[m, "i"], g.loc[m, "r"]))
    return rel, grades


def paired_bootstrap(a, b, n_boot=2000, seed=SEED, alpha=0.05):
    """Paired bootstrap ระดับผู้ใช้ (2.10.2): สุ่มซ้ำ "ผู้ใช้" (ไม่ใช่ปฏิสัมพันธ์ — ตัวอย่าง 2.17) แล้วดู CI ของผลต่าง b−a
    คืน dict {diff, ci_low, ci_high, p_value(สองด้าน โดยประมาณ), significant}"""
    a, b = np.asarray(a, float), np.asarray(b, float)
    rng = np.random.default_rng(seed)
    n = len(a)
    d = b - a
    boots = np.array([d[rng.integers(0, n, n)].mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    return {"diff": float(d.mean()), "ci_low": float(lo), "ci_high": float(hi), "p_value": float(min(1.0, p)),
            "significant": bool(lo > 0 or hi < 0), "n_users": n}


def holm_bonferroni(pvals, alpha=0.05):
    """Holm–Bonferroni (2.10.4): เรียง p จากน้อยไปมาก เทียบกับ α/(m−k+1) ทีละขั้น หยุดเมื่อไม่ผ่านครั้งแรก
    คืน list ของ bool (ปฏิเสธ H0 หรือไม่) ตามลำดับเดิม"""
    m = len(pvals)
    order = np.argsort(pvals)
    reject = [False] * m
    for k, idx in enumerate(order):
        if pvals[idx] <= alpha / (m - k):
            reject[idx] = True
        else:
            break
    return reject


def seed_variance(run_fn, seeds, metric="nDCG@K"):
    """ความผันผวนจาก seed อย่างเดียว (2.10.1): run_fn(seed) → summary dict ; คืนตาราง + mean±sd
    หากผลต่างระหว่างสองโมเดลเล็กกว่า sd ของ seed ก็ยังสรุปไม่ได้ว่าใครดีกว่า"""
    rows = [{"seed": s, **run_fn(s)} for s in seeds]
    df = pd.DataFrame(rows)
    return df, float(df[metric].mean()), float(df[metric].std(ddof=1)) if len(df) > 1 else 0.0


class ProtocolCard(dict):
    """Evaluation Protocol Card (นิยาม 2.9 / ตาราง 2.15–2.16): บันทึกทุกข้อกำหนดของการทดลอง "ก่อน" ดูผล
    ใช้เป็น dict ธรรมดา แล้วเรียก .show() เพื่อพิมพ์เป็นตาราง 11 ขั้น"""
    FIELDS = ["1 เป้าหมายผู้ใช้/ธุรกิจ", "2 Primary Metric", "3 Guardrail Metrics", "4 สูตรและค่า K / นิยามความเกี่ยวข้อง",
              "5 การแบ่งข้อมูล", "6 การป้องกันรั่วไหล", "7 Candidate Set", "8 Negative Sampling",
              "9 Accuracy + Beyond-Accuracy", "10 ความไม่แน่นอน", "11 ตรวจกับผู้ใช้จริง"]
    def show(self):
        rows = [{"ขั้น": f, "ค่าที่กำหนด": self.get(f, "—")} for f in self.FIELDS]
        for k, v in self.items():
            if k not in self.FIELDS:
                rows.append({"ขั้น": k, "ค่าที่กำหนด": v})
        df = pd.DataFrame(rows)
        try:
            from IPython.display import display
            display(df.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}).hide(axis="index"))
        except Exception:
            print(df.to_string(index=False))


# =========================================================================================
# ส่วนที่ 11  การนำเสนอผล ตาราง กราฟ รายงานผู้ใช้ และฟอร์ม GUI
# =========================================================================================
ACC_COLS = ["P@K", "R@K", "F1@K", "HR@K", "MRR@K", "MAP@K", "nDCG@K", "AUC"]
BEYOND_COLS = ["Coverage@K", "Gini", "LongTail@K", "ILD@K", "Novelty@K", "CalibErr"]


def results_table(summaries, cols=None, K=None):
    """รวม summary dict ของหลายโมเดลเป็นตารางเดียว (แถว = โมเดล) ; แทน @K ด้วยค่า K จริงเมื่อระบุ"""
    df = pd.DataFrame(summaries).T
    cols = cols or [c for c in ACC_COLS + BEYOND_COLS + ["n_users"] if c in df.columns]
    df = df[cols]
    if "n_users" in df.columns:
        df["n_users"] = df["n_users"].astype(int)
    if K is not None:
        df.columns = [c.replace("@K", f"@{K}") for c in df.columns]
    return df


def show_df(df, caption=None, fmt="{:.4f}"):
    """แสดง DataFrame แบบสวยใน notebook (fallback เป็น print เมื่อไม่มี IPython)"""
    try:
        from IPython.display import display
        num_cols = [c for c in df.columns if pd.api.types.is_float_dtype(df[c])]
        sty = df.style.format(fmt, na_rep="—", subset=num_cols) if num_cols else df.style
        if caption:
            sty = sty.set_caption(caption)
        display(sty)
    except Exception:
        if caption:
            print(caption)
        print(df.to_string())


def plot_metric_bars(table, metrics, title="", ncols=4, figsize=None, highlight_best=True):
    """กราฟแท่งเทียบโมเดล: หนึ่งกราฟย่อยต่อหนึ่งตัวชี้วัด (ระบายแท่งที่ดีที่สุดเข้มกว่า)
    ตัวชี้วัดที่ "ยิ่งน้อยยิ่งดี" (Gini, CalibErr, MAE, RMSE) จะทำเครื่องหมาย ↓ ในชื่อ"""
    import matplotlib.pyplot as plt
    metrics = [m for m in metrics if m in table.columns]
    n = len(metrics)
    if n == 0:
        return
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize or (4.2 * ncols, 3.2 * nrows))
    axes = np.atleast_1d(axes).ravel()
    lower_better = {"Gini", "CalibErr", "MAE", "RMSE"}
    for ax, m in zip(axes, metrics):
        vals = table[m].astype(float)
        lb = any(m.startswith(x) for x in lower_better)
        best = vals.idxmin() if lb else vals.idxmax()
        colors = ["#2E6B4A" if (highlight_best and idx == best) else "#9DBFA9" for idx in table.index]
        ax.bar(range(len(vals)), vals.values, color=colors)
        ax.set_xticks(range(len(vals)))
        ax.set_xticklabels(table.index, rotation=45, ha="right", fontsize=8)
        ax.set_title(m + (" ↓" if lb else " ↑"), fontsize=10)
        for k, v in enumerate(vals.values):
            if not np.isnan(v):
                ax.text(k, v, f"{v:.3f}", ha="center", va="bottom", fontsize=7)
        ax.grid(axis="y", alpha=0.3)
    for ax in axes[n:]:
        ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    plt.show()


def plot_tradeoff(table, x="nDCG@K", y="ILD@K", title="Accuracy เทียบ Beyond-Accuracy"):
    """แผนภาพกระจาย: แกน x = accuracy, แกน y = beyond-accuracy — ดู "การแลกเปลี่ยน" (3.10.5, 3.20.1 ข้อ 9)"""
    import matplotlib.pyplot as plt
    if x not in table.columns or y not in table.columns:
        return
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for name, row in table.iterrows():
        ax.scatter(row[x], row[y], s=60, color="#2E6B4A")
        ax.annotate(name, (row[x], row[y]), textcoords="offset points", xytext=(5, 4), fontsize=8)
    ax.set_xlabel(x); ax.set_ylabel(y); ax.set_title(title); ax.grid(alpha=0.3)
    plt.show()


def side_by_side_topk(recs_by_model, u, item_title, K=5, scores_by_model=None):
    """ตารางเทียบรายการแนะนำ Top-K ของผู้ใช้ u จากหลายโมเดล (คอลัมน์ = โมเดล) — ตัวอย่างผลลัพธ์บนชุดตรวจสอบ"""
    cols = {}
    for name, recs in recs_by_model.items():
        r = recs.get(u, [])[:K]
        cells = []
        for it in r:
            t = item_title(it)
            if scores_by_model and name in scores_by_model and scores_by_model[name].get(u) is not None:
                t += f" ({scores_by_model[name][u][it]:.3f})"
            cells.append(t)
        cells += [""] * (K - len(cells))
        cols[name] = cells
    return pd.DataFrame(cols, index=[f"อันดับ {k}" for k in range(1, K + 1)])


def mark_hits(df_topk, rel_titles):
    """ใส่เครื่องหมาย ✔ หน้ารายการที่อยู่ในของจริง (ground truth) เพื่ออ่านผลด้วยตา"""
    def f(x):
        if not x:
            return x
        base = re.sub(r" \(-?\d+\.\d+\)$", "", x)      # ตัดเฉพาะคะแนนท้ายข้อความ ไม่ตัดวงเล็บในชื่อเรื่อง เช่น "Toy Story (1995)"
        return ("✔ " if base in rel_titles else "   ") + x
    return df_topk.applymap(f) if hasattr(df_topk, "applymap") else df_topk.map(f)


def user_report(u, train_hist, test_df, recs_by_model, item_title, item_content=None, K=5, like_thr=4):
    """รายงานผู้ใช้หนึ่งคน 3 ส่วน: ① ประวัติ (ชุดฝึก) ② ของจริงในชุดที่กันไว้ ③ Top-K ของแต่ละโมเดล (✔ = ตรงของจริง)
    item_content(i) → str เนื้อหาของรายการ (ถ้ามี) ; คืน dict ของ DataFrame"""
    hist = train_hist.get(u, {})
    h = pd.DataFrame([{"item": item_title(i), "rating": r, **({"เนื้อหา": item_content(i)} if item_content else {})}
                      for i, r in sorted(hist.items(), key=lambda x: -x[1])])
    te = test_df[test_df["u"] == u]
    rel_titles = set(item_title(i) for i in te.loc[te["r"] >= like_thr, "i"])
    g = pd.DataFrame([{"item": item_title(i), "rating": r, "เกี่ยวข้อง": "✔" if r >= like_thr else "",
                       **({"เนื้อหา": item_content(i)} if item_content else {})}
                      for i, r in zip(te["i"], te["r"])])
    top = mark_hits(side_by_side_topk(recs_by_model, u, item_title, K), rel_titles)
    return {"history": h, "ground_truth": g, "recommended": top}


def build_recommender_form(user_ids, model_names, recommend_fn, user_info_fn, K_default=5, K_max=20,
                           title="ตัวอย่างการประยุกต์ใช้: ฟอร์มแนะนำภาพยนตร์"):
    """ฟอร์ม GUI ด้วย ipywidgets (มีใน Colab อยู่แล้ว)
    recommend_fn(user_id, model_name, K) → DataFrame รายการที่แนะนำ (พร้อมเนื้อหา/คะแนน)
    user_info_fn(user_id)                → DataFrame ข้อมูล/ประวัติของผู้ใช้
    """
    try:
        import ipywidgets as W
        from IPython.display import display, HTML, clear_output
    except Exception:
        print("ไม่มี ipywidgets — ติดตั้งด้วย pip install ipywidgets")
        return None
    w_user = W.Dropdown(options=list(user_ids), description="ผู้ใช้", layout=W.Layout(width="260px"))
    w_model = W.Dropdown(options=list(model_names), description="โมเดล", layout=W.Layout(width="320px"))
    w_k = W.IntSlider(value=K_default, min=1, max=K_max, description="K", continuous_update=False)
    btn = W.Button(description="แนะนำ", button_style="success", icon="play")
    out = W.Output()

    def run(_=None):
        with out:
            clear_output()
            display(HTML(f"<h4 style='color:#2E6B4A'>ข้อมูลผู้ใช้ {w_user.value}</h4>"))
            display(user_info_fn(w_user.value))
            display(HTML(f"<h4 style='color:#2E6B4A'>รายการที่แนะนำ Top-{w_k.value} โดย {w_model.value}</h4>"))
            display(recommend_fn(w_user.value, w_model.value, w_k.value))
    btn.on_click(run)
    w_user.observe(lambda c: run() if c["name"] == "value" else None, names="value")
    w_model.observe(lambda c: run() if c["name"] == "value" else None, names="value")
    box = W.VBox([W.HTML(f"<h3 style='color:#2E6B4A'>{title}</h3>"), W.HBox([w_user, w_model, w_k, btn]), out])
    display(box)
    run()
    return box

# ===================== cbrec_core (ฝัง) — จบ =====================

In [ ]:
set_seed(SEED)
print("การตั้งค่า:", dict(FAST_MODE=FAST_MODE, SEED=SEED, K=K, LIKE_THR=LIKE_THR, K_CORE=K_CORE, N_TUNE=N_TUNE))

## ขั้น ① · โหลด · ตรวจสอบ · ทำความสะอาดข้อมูล
**ข้อสังเกต** — การเตรียมข้อมูลคือขั้นที่ "ผิดแล้วทุกขั้นถัดไปผิดตาม" (ตาราง 3.66 ลำดับ 1) จึงต้องมีรายการตรวจ (audit) ที่รันซ้ำได้

In [ ]:
def load_movielens(data_dir=DATA_DIR):
    files = ["ratings.csv", "movies.csv", "tags.csv", "links.csv"]
    os.makedirs(data_dir, exist_ok=True)
    if not all(os.path.exists(os.path.join(data_dir, f)) for f in files):
        try:
            url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
            z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url, timeout=60).read()))
            for f in files:
                open(os.path.join(data_dir, f), "wb").write(z.read(f"ml-latest-small/{f}"))
        except Exception as e:
            print("GroupLens ไม่สำเร็จ:", str(e)[:60], "→ ใช้ mirror บน GitHub")
            for f in files:
                urllib.request.urlretrieve("https://raw.githubusercontent.com/smanihwr/ml-latest-small/master/" + f, os.path.join(data_dir, f))
    return (pd.read_csv(os.path.join(data_dir, f)) for f in files[:3])

ratings_raw, movies, tags = load_movielens()

def audit(ratings, movies, tags):
    """รายการตรวจข้อมูลก่อนใช้ — ทุกข้อคือ 'อาการ' ที่เคยทำให้การทดลองผิดโดยไม่มี error"""
    rep = {}
    rep["แถวซ้ำ (u,i)"] = int(ratings.duplicated(["userId", "movieId"]).sum())
    rep["คะแนนนอกช่วง 0.5–5"] = int(((ratings.rating < 0.5) | (ratings.rating > 5)).sum())
    rep["timestamp เป็นศูนย์/ว่าง"] = int((ratings.timestamp.fillna(0) <= 0).sum())
    rep["movieId ในคะแนนที่ไม่มีใน movies"] = int((~ratings.movieId.isin(movies.movieId)).sum())
    rep["movieId ซ้ำใน movies"] = int(movies.movieId.duplicated().sum())
    rep["ชื่อเรื่องซ้ำ (คนละ movieId)"] = int(movies.title.duplicated().sum())
    rep["เรื่องที่ไม่มีแนว"] = int(movies.genres.str.startswith("(").sum())
    rep["tag ว่าง/NaN"] = int(tags.tag.isna().sum())
    rep["ผู้ใช้ที่ให้คะแนน < 20"] = int((ratings.groupby("userId").size() < 20).sum())
    return pd.Series(rep, name="จำนวน")
show_df(audit(ratings_raw, movies, tags).to_frame(), "รายการตรวจข้อมูล (audit)", fmt="{}")
# ทำความสะอาดตามผลตรวจ: ตัดคู่ซ้ำ (เก็บล่าสุด) · ตัดคะแนนที่ไม่มีรายการ · tag NaN → ว่าง
ratings_raw = ratings_raw.sort_values("timestamp").drop_duplicates(["userId", "movieId"], keep="last")
ratings_raw = ratings_raw[ratings_raw.movieId.isin(movies.movieId)]
tags["tag"] = tags["tag"].fillna("").astype(str)
print("หลังทำความสะอาด:", len(ratings_raw), "แถว")

In [ ]:
# ---------- เขตข้อมูลเนื้อหาหลายเขต (ใช้ในขั้น ③): title · genre · tag · decade ----------
import re as _re
def split_title(t):
    m = _re.match(r"^(.*?)\s*\((\d{4})\)\s*$", t)
    title, year = (m.group(1), int(m.group(2))) if m else (t, np.nan)
    return _re.sub(r"^(.*), (The|A|An)$", r"\2 \1", title.strip()), year
movies[["title_clean", "year"]] = movies["title"].apply(lambda t: pd.Series(split_title(t)))
movies["genre_tok"] = movies["genres"].apply(lambda g: " ".join(x.replace("-", "").replace(" ", "").lower() for x in g.split("|") if not x.startswith("(")))
tag_text = tags.groupby("movieId")["tag"].apply(lambda s: " ".join(x.lower() for x in s)).rename("tag_text")
movies = movies.merge(tag_text, on="movieId", how="left").fillna({"tag_text": ""})
def decade_token(y):
    """โทเคนทศวรรษที่เป็นตัวอักษรล้วน (ตัวตัดคำ [a-z]+ จะไม่แยกตัวเลขออก): 1990 → 'decadebjja' (b=1, j=9, a=0)"""
    if pd.isna(y) or y <= 0: return ""
    return "decade" + "".join("abcdefghij"[int(c)] for c in str(int(y) // 10 * 10))
movies["decade_tok"] = movies["year"].apply(decade_token)   # โทเคนพิเศษต่อทศวรรษ
N_ITEMS = len(movies)
GENRES = sorted({g for gs in movies["genres"] for g in gs.split("|") if not g.startswith("(")})
G = np.array([[1.0 if g in gs.split("|") else 0.0 for g in GENRES] for gs in movies["genres"]]); Gn = l2_normalize_rows(G)
def title_of(i): return f"{movies.title_clean[i]} ({int(movies.year[i]) if pd.notna(movies.year[i]) else '—'})"

# ---------- k-core ก่อนแบ่ง + (FAST_MODE) สุ่มผู้ใช้ ----------
df_all, u_map, i_map = encode_ids(ratings_raw, "userId", "movieId", items_df=movies, item_id_col="movieId")
df_core = kcore_filter(df_all[["userId", "movieId", "u", "i", "r", "timestamp"]], K_CORE, K_CORE)
if FAST_MODE:
    keep = np.random.default_rng(SEED).choice(df_core.u.unique(), 200, replace=False); df_core = df_core[df_core.u.isin(keep)]
N_USERS = int(df_all.u.max()) + 1
print(f"หลัง {K_CORE}-core: {len(df_core)} แถว · ผู้ใช้ {df_core.u.nunique()} · รายการ {df_core.i.nunique()} / คลัง {N_ITEMS}")

## ขั้น ② · แบ่ง Train / Valid-A / Valid-B / Test ตามเวลาในผู้ใช้
* **Valid-A** ใช้ปรับจูน hyperparameter (แตะได้หลายร้อยครั้ง) · **Valid-B** ใช้เลือกโมเดลสุดท้ายและปรับเทียบ (แตะไม่กี่ครั้ง) ·
  **Test** แตะครั้งเดียว — สองชุดตรวจสอบช่วยตรวจว่าการปรับจูน "พอดีกับสัญญาณรบกวนของ Valid-A" หรือไม่ (ตาราง 3.68 แถว 7)

In [ ]:
rest, test = split_per_user_temporal(df_core, 0.2)
rest2, valid_b = split_per_user_temporal(rest, 0.1 / 0.8)
train, valid_a = split_per_user_temporal(rest2, 0.1 / 0.7)
for nm, d in [("train", train), ("valid_a", valid_a), ("valid_b", valid_b), ("test", test)]:
    print(f"{nm:8s}: {len(d):6d} แถว · ผู้ใช้ {d.u.nunique()}")
assert not (set(zip(train.u, train.i)) & set(zip(test.u, test.i)))
train_hist = hist_from(train); relA, grA = relevant_from(valid_a, LIKE_THR); relB, grB = relevant_from(valid_b, LIKE_THR)
users_A = sorted(u for u, s in relA.items() if s and u in train_hist); users_B = sorted(u for u, s in relB.items() if s and u in train_hist)
rng = np.random.default_rng(SEED)
users_tune = sorted(rng.choice(users_A, size=min(N_TUNE, len(users_A)), replace=False).tolist())   # กลุ่มย่อยสำหรับปรับจูน
pop = ItemPopularity(train, N_ITEMS, N_USERS); head = head_items_from_train(pop.count, N_ITEMS)
POOL = set(int(i) for i in train.i.unique())
print(f"ผู้ใช้ประเมินได้: Valid-A {len(users_A)} (ปรับจูนบน {len(users_tune)}) · Valid-B {len(users_B)} · pool {len(POOL)} เรื่อง")
pop_tb = 1e-6 * minmax(pop.p)     # ตัวตัดสินรองเมื่อคะแนนเสมอ = ความนิยม (3.10.2)

def quick_eval(score_fn, users=None, rel=None, grades=None, rerank_fn=None, hist=None, pool=POOL, **kw):
    """ตัวห่อ evaluate_ranking สำหรับการปรับจูน — คืน summary dict (ค่าตั้งต้น = กลุ่มย่อยของ Valid-A)"""
    pu, sm, recs = evaluate_ranking(score_fn, users or users_tune, hist or train_hist, rel or relA, N_ITEMS, K=K, item_feat=G, pop=kw.get("pop_", pop),
                                    head=kw.get("head_", head), genre_mat=G, rel_grades=grades or grA, rerank_fn=rerank_fn, seed=SEED, pool=pool)
    return sm, pu, recs

## ขั้น ③ · เขตข้อมูลเนื้อหาหลายเขตและน้ำหนักเขตข้อมูล (แนวคิด BM25F, หัวข้อ 3.11.5)
เอกสารของรายการ = title × w_t + genre × w_g + tag × w_tag + decade × w_d โดย "ทำซ้ำ" โทเคนตามน้ำหนัก (จำนวนเต็ม) —
วิธีอย่างง่ายที่ทำให้ TF-IDF/BM25 มาตรฐานถ่วงน้ำหนักเขตข้อมูลได้ ค่าที่ดีที่สุดจะถูกเลือกในขั้น ④

In [ ]:
def build_text(w_title=1, w_genre=2, w_tag=1, w_decade=0):
    parts = [(movies["title_clean"] + " ") * w_title, (movies["genre_tok"] + " ") * w_genre, (movies["tag_text"] + " ") * w_tag, (movies["decade_tok"] + " ") * w_decade]
    doc = parts[0]
    for x in parts[1:]:
        doc = doc + x
    return list(doc.str.strip())
print("ตัวอย่างเอกสารของรายการ (w_genre=2, w_decade=1):", build_text(1, 2, 1, 1)[0][:120])

## ขั้น ④ · ปรับจูนการแทนเนื้อหา (ลำดับ 1–4 ของตาราง 3.66) บน Valid-A ด้วยโมเดล TFIDF-centroid
Primary = nDCG@10 · บันทึกทุก config ลง `tuning_log` · **ข้อสังเกต**: ปรับ "ทีละกลุ่ม" ตามลำดับ ไม่ใช่กริดเต็มทุกตัวพร้อมกัน
(กริดเต็ม 4×3×5×3×… จะระเบิดและ overfit ชุดตรวจสอบ) — วิธีนี้เรียกว่า coordinate-wise search

In [ ]:
tuning_log = []
def log_result(stage, cfg, sm, t):
    row = {"ขั้น": stage, **cfg, "nDCG@10": sm["nDCG@K"], "HR@10": sm["HR@K"], "Coverage@10": sm["Coverage@K"], "ILD@10": sm["ILD@K"], "Novelty@10": sm["Novelty@K"], "วินาที": round(t, 1)}
    tuning_log.append(row); return row

def make_tfidf_scorer(vec, X, method="centroid", **pkw):
    def fn(u):
        p = build_profile(X, train_hist.get(u, {}), method, LIKE_THR, DISLIKE_THR, **pkw)
        return match_scores(X, p) + pop_tb
    return fn

# 4.1 น้ำหนักเขตข้อมูล (ลำดับ 2: ความครบถ้วนของเนื้อหา)
rows = []
field_grid = [(1, 1, 1, 0), (1, 2, 1, 0), (1, 3, 1, 0), (1, 2, 2, 0), (0, 2, 1, 0), (1, 2, 1, 1), (1, 3, 2, 1)]
for wt, wg, wtag, wd in field_grid:
    t0 = time.time()
    vec = TextVectorizer(min_df=2, max_df=0.85); X = vec.fit_transform(build_text(wt, wg, wtag, wd))
    sm, _, _ = quick_eval(make_tfidf_scorer(vec, X))
    rows.append(log_result("④ field weights", {"w_title": wt, "w_genre": wg, "w_tag": wtag, "w_decade": wd, "vocab": len(vec.vocab_)}, sm, time.time() - t0))
res_field = pd.DataFrame(rows).drop(columns="ขั้น"); show_df(res_field, "④.1 น้ำหนักเขตข้อมูล — nDCG@10 บนกลุ่มปรับจูน (Valid-A)")
BEST_FIELD = tuple(res_field.loc[res_field["nDCG@10"].idxmax(), ["w_title", "w_genre", "w_tag", "w_decade"]].astype(int))
print("เลือก field weights =", BEST_FIELD)
# ข้อสังเกต: ถ้าตัดชื่อเรื่องออก (w_title=0) แล้วดีขึ้น แปลว่าชื่อเรื่องเป็นสัญญาณรบกวน (คำในชื่อไม่ค่อยซ้ำข้ามเรื่อง)
#            decade ช่วยจับ "รสนิยมตามยุค" ซึ่งแนวอย่างเดียวไม่บอก

In [ ]:
# 4.2 คุมคลังศัพท์ (ลำดับ 3): min_df × ngram  และ 4.3 รูปแบบถ่วงน้ำหนัก (ลำดับ 4): TF × IDF
TEXT = build_text(*BEST_FIELD)
rows = []
for min_df, ngram in itertools.product([1, 2, 5], [(1, 1), (1, 2)]):
    t0 = time.time(); vec = TextVectorizer(min_df=min_df, max_df=0.85, ngram_range=ngram); X = vec.fit_transform(TEXT)
    sm, _, _ = quick_eval(make_tfidf_scorer(vec, X))
    rows.append(log_result("④ vocab", {"min_df": min_df, "ngram": str(ngram), "vocab": len(vec.vocab_)}, sm, time.time() - t0))
res_vocab = pd.DataFrame(rows).drop(columns="ขั้น"); show_df(res_vocab, "④.2 min_df × ngram — ดูตัวชี้วัด 'คู่กับ' ขนาดคลังศัพท์ (ตาราง 3.66 ลำดับ 3)")
b = res_vocab.loc[res_vocab["nDCG@10"].idxmax()]; BEST_MIN_DF, BEST_NGRAM = int(b["min_df"]), eval(b["ngram"])
rows = []
for tf, idf in itertools.product(["raw", "log", "binary", "aug"], ["smooth", "standard", "bm25", "none"]):
    t0 = time.time(); vec = TextVectorizer(tf=tf, idf=idf, min_df=BEST_MIN_DF, max_df=0.85, ngram_range=BEST_NGRAM); X = vec.fit_transform(TEXT)
    sm, _, _ = quick_eval(make_tfidf_scorer(vec, X))
    rows.append(log_result("④ weighting", {"tf": tf, "idf": idf}, sm, time.time() - t0))
res_w = pd.DataFrame(rows).drop(columns="ขั้น")
show_df(res_w.pivot(index="tf", columns="idf", values="nDCG@10"), "④.3 TF × IDF scheme → nDCG@10 (ตาราง 3.20/3.24: TF-IDF เป็นตระกูลของสูตร)")
b = res_w.loc[res_w["nDCG@10"].idxmax()]; BEST_TF, BEST_IDF = b["tf"], b["idf"]
print(f"เลือก min_df={BEST_MIN_DF}, ngram={BEST_NGRAM}, tf={BEST_TF}, idf={BEST_IDF}")
# ข้อสังเกต: idf='none' (TF อย่างเดียว) มักแย่ชัดเจน = หลักฐานว่า IDF จำเป็น ; binary กับ log มักใกล้กันเพราะเอกสารสั้น (avgdl ~8)
fig, ax = plt.subplots(figsize=(6, 3.5)); im = ax.imshow(res_w.pivot(index="tf", columns="idf", values="nDCG@10").values, cmap="Greens")
ax.set_xticks(range(4)); ax.set_xticklabels(["smooth", "standard", "bm25", "none"]); ax.set_yticks(range(4)); ax.set_yticklabels(["raw", "log", "binary", "aug"])
ax.set_xlabel("IDF"); ax.set_ylabel("TF"); ax.set_title("nDCG@10 by TF × IDF scheme"); plt.colorbar(im); plt.tight_layout(); plt.show()

# การแทนเนื้อหาที่เลือก (ใช้ต่อทุกขั้น)
VEC = TextVectorizer(tf=BEST_TF, idf=BEST_IDF, min_df=BEST_MIN_DF, max_df=0.85, ngram_range=BEST_NGRAM); X = VEC.fit_transform(TEXT)
C_counts = TextVectorizer(tf="raw", idf="none", norm=None, min_df=BEST_MIN_DF, max_df=0.85, ngram_range=BEST_NGRAM).fit_transform(TEXT)
TOKENS = VEC._train_tokens
print("X:", X.shape)

## ขั้น ⑤ · ปรับจูน BM25: k₁ × b และจำนวนคำในข้อคำถาม (ลำดับ 5)

In [ ]:
rows = []
k1_grid, b_grid = ([0.5, 1.2, 1.5, 2.0, 3.0], [0.0, 0.5, 0.75, 1.0]) if not FAST_MODE else ([0.8, 1.5, 3.0], [0.0, 0.75])
for k1, b_ in itertools.product(k1_grid, b_grid):
    t0 = time.time(); bm = BM25(k1=k1, b=b_).fit(TOKENS)
    def fn(u, bm=bm):
        p = build_profile(X, train_hist.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR); top = VEC.top_terms(p, 20)
        return bm.score_query([t for t, _ in top], dict(top)) + pop_tb if top else None
    sm, _, _ = quick_eval(fn); rows.append(log_result("⑤ BM25", {"k1": k1, "b": b_, "n_query": 20}, sm, time.time() - t0))
res_bm = pd.DataFrame(rows).drop(columns="ขั้น")
piv = res_bm.pivot(index="k1", columns="b", values="nDCG@10"); show_df(piv, "⑤ BM25 k₁ × b → nDCG@10 (ภาพ 3.21: 'เปลี่ยนระยะห่างมากกว่าเปลี่ยนลำดับ')")
fig, ax = plt.subplots(figsize=(6, 3.5)); im = ax.imshow(piv.values, cmap="Greens"); ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns); ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
ax.set_xlabel("b"); ax.set_ylabel("k1"); ax.set_title("BM25 nDCG@10"); plt.colorbar(im); plt.tight_layout(); plt.show()
bb = res_bm.loc[res_bm["nDCG@10"].idxmax()]; BEST_K1, BEST_B = float(bb["k1"]), float(bb["b"])
rows = []
for nq in [5, 10, 20, 40, 80, 160]:
    t0 = time.time(); bm = BM25(k1=BEST_K1, b=BEST_B).fit(TOKENS)
    def fn(u, bm=bm, nq=nq):
        p = build_profile(X, train_hist.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR); top = VEC.top_terms(p, nq)
        return bm.score_query([t for t, _ in top], dict(top)) + pop_tb if top else None
    sm, _, _ = quick_eval(fn); rows.append(log_result("⑤ BM25 n_query", {"k1": BEST_K1, "b": BEST_B, "n_query": nq}, sm, time.time() - t0))
res_nq = pd.DataFrame(rows).drop(columns="ขั้น"); show_df(res_nq, "⑤ จำนวนคำในข้อคำถาม (3.11.7: BM25 ไวต่อ n มาก)")
BEST_NQ = int(res_nq.loc[res_nq["nDCG@10"].idxmax(), "n_query"]); BM = BM25(k1=BEST_K1, b=BEST_B).fit(TOKENS)
print(f"เลือก k1={BEST_K1}, b={BEST_B}, n_query={BEST_NQ}")
# ข้อสังเกต: n_query ยิ่งมากยิ่งใกล้ "โคไซน์ระหว่างโปรไฟล์ทั้งเวกเตอร์กับเอกสาร" (ต่างกันที่การอิ่มตัว) — ถ้าค่าที่ดีที่สุดคือค่าสูงสุดของกริด
#            แปลว่ากริดยังไม่กว้างพอ ต้องขยาย (หลักทั่วไปของการปรับจูน: ค่าที่เลือกไม่ควรอยู่ที่ขอบกริด)
# ข้อสังเกต: b เล็ก (ไม่ปรับความยาว) มักดีกว่าในคลังที่เอกสาร 'ยาว' เพราะมี tag มาก = มีข้อมูลมาก ไม่ใช่เพราะเยิ่นเย้อ

## ขั้น ⑥ · ปรับจูนโปรไฟล์ (ลำดับ 6): เกณฑ์ชอบ · Rocchio β/γ · Naive Bayes α

In [ ]:
rows = []
for like_thr in [3.5, 4.0, 4.5]:
    t0 = time.time()
    def fn(u, lt=like_thr): return match_scores(X, build_profile(X, train_hist.get(u, {}), "centroid", lt, DISLIKE_THR)) + pop_tb
    sm, _, _ = quick_eval(fn); rows.append(log_result("⑥ like_thr", {"like_thr(profile)": like_thr}, sm, time.time() - t0))
for beta, gamma in [(1.0, 0.0), (0.9, 0.1), (0.8, 0.2), (0.7, 0.3), (0.5, 0.5)]:
    t0 = time.time()
    def fn(u, b=beta, g=gamma): return match_scores(X, build_profile(X, train_hist.get(u, {}), "rocchio", LIKE_THR, DISLIKE_THR, beta=b, gamma=g)) + pop_tb
    sm, _, _ = quick_eval(fn); rows.append(log_result("⑥ Rocchio", {"beta": beta, "gamma": gamma}, sm, time.time() - t0))
for alpha in [0.01, 0.1, 0.5, 1.0]:
    t0 = time.time()
    def fn(u, a=alpha):
        s = ClassifierProfile("nb", alpha=a, like_thr=LIKE_THR, dislike_thr=DISLIKE_THR, min_examples=3).fit(C_counts, train_hist.get(u, {})).score(C_counts)
        return (s if s is not None else match_scores(X, build_profile(X, train_hist.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR))) + pop_tb
    sm, _, _ = quick_eval(fn); rows.append(log_result("⑥ NB alpha", {"nb_alpha": alpha}, sm, time.time() - t0))
res_prof = pd.DataFrame(rows); show_df(res_prof.round(4).fillna(""), "⑥ พารามิเตอร์ของโปรไฟล์ (สังเกต ILD คู่กับ nDCG ตามตาราง 3.66)", fmt="{}")
r_ro = res_prof[res_prof["ขั้น"] == "⑥ Rocchio"]; bb = r_ro.loc[r_ro["nDCG@10"].idxmax()]; BEST_BETA, BEST_GAMMA = float(bb["beta"]), float(bb["gamma"])
r_nb = res_prof[res_prof["ขั้น"] == "⑥ NB alpha"]; BEST_ALPHA = float(r_nb.loc[r_nb["nDCG@10"].idxmax(), "nb_alpha"])
print(f"เลือก Rocchio β={BEST_BETA}, γ={BEST_GAMMA} · NB α={BEST_ALPHA}")
# ข้อสังเกต: γ ใหญ่ (0.5) มักแย่ลง — 'กฎ β>γ' ; NB α เล็ก = เชื่อคำที่พบน้อยมากเกินไป (คำเดียวพลิกผล, ตัวอย่าง 3.12)

## ขั้น ⑦ · ปรับจูนมิติแฝง LSA (ลำดับ 7)

In [ ]:
rows = []; LSA_CACHE = {}
for k in ([20, 50, 100, 200, 400] if not FAST_MODE else [50, 200]):
    t0 = time.time(); Z = LSA(k=k, seed=SEED).fit(X).transform(X); LSA_CACHE[k] = Z
    def fn(u, Z=Z): return match_scores(Z, build_profile(Z, train_hist.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR)) + pop_tb
    sm, _, _ = quick_eval(fn); rows.append(log_result("⑦ LSA k", {"lsa_k": k}, sm, time.time() - t0))
res_lsa = pd.DataFrame(rows).drop(columns="ขั้น"); show_df(res_lsa, "⑦ LSA k → nDCG@10 (ค่าแนะนำ 100–300 ในตาราง 3.67)")
BEST_LSA_K = int(res_lsa.loc[res_lsa["nDCG@10"].idxmax(), "lsa_k"]); Z = LSA_CACHE[BEST_LSA_K]; print("เลือก LSA k =", BEST_LSA_K)
fig, ax = plt.subplots(figsize=(5, 3)); ax.plot(res_lsa["lsa_k"], res_lsa["nDCG@10"], "o-", color="#2E6B4A"); ax.set_xscale("log"); ax.set_xlabel("LSA k"); ax.set_ylabel("nDCG@10"); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## ขั้น ⑧ · ปรับจูนการผสาน (ลำดับ 8): convex weight ระหว่าง lexical (BM25) กับ semantic (LSA) และ RRF k

In [ ]:
def s_tfidf(u):  return match_scores(X, build_profile(X, train_hist.get(u, {}), "rocchio", LIKE_THR, DISLIKE_THR, beta=BEST_BETA, gamma=BEST_GAMMA))
def s_genre(u):  return match_scores(Gn, build_profile(G, train_hist.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR))
def s_lsa(u):    return match_scores(Z, build_profile(Z, train_hist.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR))
def s_bm25(u):
    p = build_profile(X, train_hist.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR); top = VEC.top_terms(p, BEST_NQ)
    return BM.score_query([t for t, _ in top], dict(top)) if top else np.zeros(N_ITEMS)
def s_nb(u):
    s = ClassifierProfile("nb", alpha=BEST_ALPHA, like_thr=LIKE_THR, dislike_thr=DISLIKE_THR, min_examples=3).fit(C_counts, train_hist.get(u, {})).score(C_counts)
    return s if s is not None else s_tfidf(u)
rows = []
for w in [0.0, 0.3, 0.5, 0.7, 1.0]:
    t0 = time.time()
    def fn(u, w=w): return convex_fusion([s_bm25(u), s_lsa(u)], [w, 1 - w]) + pop_tb
    sm, _, _ = quick_eval(fn); rows.append(log_result("⑧ convex w", {"w_bm25": w}, sm, time.time() - t0))
for kr in [10, 60, 200]:
    t0 = time.time()
    def fn(u, kr=kr): return rrf_fusion([s_bm25(u), s_lsa(u), s_genre(u), s_nb(u)], k=kr) + pop_tb
    sm, _, _ = quick_eval(fn); rows.append(log_result("⑧ RRF k", {"rrf_k": kr}, sm, time.time() - t0))
res_fus = pd.DataFrame(rows); show_df(res_fus.round(4).fillna(""), "⑧ การผสาน (ตัวอย่าง 3.22: ผู้ชนะเปลี่ยนตาม w) — w_bm25=1 คือ BM25 ล้วน, 0 คือ LSA ล้วน", fmt="{}")
r_c = res_fus[res_fus["ขั้น"] == "⑧ convex w"]; BEST_W = float(r_c.loc[r_c["nDCG@10"].idxmax(), "w_bm25"])
r_r = res_fus[res_fus["ขั้น"] == "⑧ RRF k"]; BEST_RRF_K = int(r_r.loc[r_r["nDCG@10"].idxmax(), "rrf_k"])
print(f"เลือก w_bm25={BEST_W} · RRF k={BEST_RRF_K}")

## ขั้น ⑨ · ปรับจูนความหลากหลาย λ ของ MMR ภายใต้ guardrail (ลำดับ 9 — "การแลกเปลี่ยน ไม่ใช่การเพิ่มคุณภาพ")
Guardrail ที่ประกาศ: เลือก λ ที่ **ILD@10 เพิ่มมากที่สุดโดย nDCG@10 ลดไม่เกิน 5% เทียบกับ λ=1**

In [ ]:
def s_hybrid(u): return rrf_fusion([s_bm25(u), s_lsa(u), s_genre(u), s_nb(u)], k=BEST_RRF_K) + pop_tb
def make_mmr(lam, hist=None, pool=POOL):
    hist = hist or train_hist
    def rerank(u, s):
        base = top_k(s, set(hist.get(u, {})), K=5 * K, candidates=pool)
        if not base: return []
        Sm = np.asarray(Gn[base] @ Gn[base].T)
        return [base[j] for j in mmr_rerank(s[base], Sm, K=K, lam=lam, pool=len(base))]
    return rerank
rows = []
for lam in [1.0, 0.9, 0.8, 0.7, 0.5, 0.3]:
    t0 = time.time(); sm, _, _ = quick_eval(s_hybrid, rerank_fn=make_mmr(lam)); rows.append(log_result("⑨ MMR λ", {"lambda": lam}, sm, time.time() - t0))
res_mmr = pd.DataFrame(rows).drop(columns="ขั้น"); show_df(res_mmr, "⑨ λ ของ MMR: nDCG ลด ↔ ILD เพิ่ม")
base_ndcg = float(res_mmr.loc[res_mmr["lambda"] == 1.0, "nDCG@10"].iloc[0])
ok = res_mmr[res_mmr["nDCG@10"] >= 0.95 * base_ndcg]; BEST_LAM = float(ok.loc[ok["ILD@10"].idxmax(), "lambda"])
print(f"เลือก λ = {BEST_LAM} (guardrail: nDCG ≥ 95% ของ λ=1 คือ ≥ {0.95*base_ndcg:.4f})")
# ข้อสังเกต: ถ้า nDCG 'เพิ่ม' เมื่อ λ ลด แปลว่าการกระจายแนวช่วยผู้ใช้ที่มีของจริงหลายแนว หรือเป็นสัญญาณรบกวนของกลุ่มปรับจูน 150 คน
#            → ตรวจซ้ำบน Valid-B (ตารางถัดไป) ก่อนเชื่อ
fig, ax = plt.subplots(figsize=(5, 3.5)); ax.plot(res_mmr["nDCG@10"], res_mmr["ILD@10"], "o-", color="#2E6B4A")
for _, r in res_mmr.iterrows(): ax.annotate(f"λ={r['lambda']}", (r["nDCG@10"], r["ILD@10"]), fontsize=8, xytext=(4, 3), textcoords="offset points")
ax.set_xlabel("nDCG@10"); ax.set_ylabel("ILD@10"); ax.set_title("MMR trade-off curve"); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# ---------- บันทึกล็อกการปรับจูนทั้งหมด + ค่าที่เลือก ----------
tune_df = pd.DataFrame(tuning_log); tune_df.to_csv(f"{ART_DIR}/tuning_log.csv", index=False)
CHOSEN = dict(field_weights=BEST_FIELD, min_df=BEST_MIN_DF, ngram=BEST_NGRAM, tf=BEST_TF, idf=BEST_IDF, bm25_k1=BEST_K1, bm25_b=BEST_B, n_query=BEST_NQ,
              rocchio_beta=BEST_BETA, rocchio_gamma=BEST_GAMMA, nb_alpha=BEST_ALPHA, lsa_k=BEST_LSA_K, w_bm25=BEST_W, rrf_k=BEST_RRF_K, mmr_lambda=BEST_LAM)
json.dump({k: (list(v) if isinstance(v, tuple) else v) for k, v in CHOSEN.items()}, open(f"{ART_DIR}/chosen_hyperparams.json", "w"), indent=1)
show_df(pd.Series(CHOSEN, name="ค่าที่เลือก").astype(str).to_frame(), f"สรุปไฮเพอร์พารามิเตอร์ที่เลือกจาก Valid-A ({len(tune_df)} การทดลอง) — บันทึกที่ {ART_DIR}/", fmt="{}")
fig, ax = plt.subplots(figsize=(10, 3.2)); ax.plot(range(len(tune_df)), tune_df["nDCG@10"], ".-", color="#2E6B4A")
for st in tune_df["ขั้น"].unique():
    i0 = tune_df.index[tune_df["ขั้น"] == st][0]; ax.axvline(i0, color="grey", lw=.5, ls="--"); ax.text(i0, ax.get_ylim()[1] * .98, st, fontsize=7, rotation=90, va="top")
ax.set_xlabel("การทดลองที่"); ax.set_ylabel("nDCG@10 (tune users)"); ax.set_title("Tuning trajectory"); plt.tight_layout(); plt.show()

## ขั้น ⑩ · โมเดลทำนายขั้นสุดท้ายสำหรับการจัดอันดับ: Learning-to-Rank แบบ stacking (pointwise) + การปรับเทียบ
* **คุณลักษณะต่อคู่ (u,i)**: คะแนน TF-IDF/Rocchio · genre · LSA · BM25 · NB · ความนิยม p(i) · ปีของเรื่อง · ความยาวประวัติ
* **ป้ายกำกับ**: 1 ถ้า (u,i) มีคะแนน ≥4 ใน Valid-A, 0 = รายการที่ไม่เคยเห็นสุ่ม (negative sampling 1:4) — ฝึกบน users_A
* **โมเดล**: Logistic Regression (pointwise, อ่านน้ำหนักได้ ตาราง 3.64) และ Gradient Boosting (ไม่เชิงเส้น)
* **ปรับเทียบ**: Platt บน Valid-B + ECE (นิยาม 3.19–3.20) เพื่อให้ตั้งเกณฑ์ "แนะนำเมื่อ P(ชอบ) > τ" ได้

In [ ]:
BASE_SCORERS = {"tfidf_rocchio": s_tfidf, "genre": s_genre, "lsa": s_lsa, "bm25": s_bm25, "nb": s_nb}
year_norm = minmax(movies["year"].fillna(movies["year"].median()).to_numpy())
def user_features(u, hist):
    """เมทริกซ์คุณลักษณะของผู้ใช้ u สำหรับทุกรายการ (n_items × F) — ปรับมาตราแต่ละคะแนนด้วย min-max ภายในผู้ใช้ (3.18.1)"""
    cols = [minmax(fn(u)) for fn in BASE_SCORERS.values()]
    cols += [pop.p, year_norm, np.full(N_ITEMS, min(len(hist.get(u, {})), 200) / 200)]
    return np.column_stack(cols)
FEAT_NAMES = list(BASE_SCORERS) + ["popularity", "year", "hist_len"]

def build_ltr_dataset(users, rel, hist, neg_ratio=4, seed=SEED):
    rng = np.random.default_rng(seed); Xs, ys, groups = [], [], []
    for u in users:
        F = user_features(u, hist); pos = list(rel[u]); seen = set(hist.get(u, {}))
        unseen = np.setdiff1d(np.array(sorted(POOL)), list(seen | rel[u])); neg = rng.choice(unseen, size=min(len(unseen), neg_ratio * len(pos)), replace=False)
        idx = pos + list(neg); Xs.append(F[idx]); ys.append(np.r_[np.ones(len(pos)), np.zeros(len(neg))]); groups += [u] * len(idx)
    return np.vstack(Xs), np.concatenate(ys), np.array(groups)
with Timer("สร้างชุดฝึก LTR จาก Valid-A"):
    XA, yA, gA = build_ltr_dataset(users_A, relA, train_hist)
print("ชุดฝึก LTR:", XA.shape, "· สัดส่วนบวก", f"{yA.mean():.2%}")
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
ltr_lr = LogisticRegression(C=1.0, max_iter=2000).fit(XA, yA)
ltr_gb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=4, random_state=SEED).fit(XA, yA)
show_df(pd.DataFrame({"น้ำหนัก (LR)": ltr_lr.coef_[0]}, index=FEAT_NAMES).T, "น้ำหนักที่ LTR-LR เรียนได้ — อ่านได้ว่าระบบใดมีอิทธิพล (บวก = ยิ่งสูงยิ่งน่าจะชอบ)")
# ข้อสังเกต (สำคัญ): น้ำหนักของ 'popularity' มักใหญ่กว่าคะแนนเนื้อหาทุกตัวหลายเท่า เพราะป้ายกำกับบวกของ MovieLens กระจุกอยู่ที่เรื่องดัง
#   → LTR จะ "เรียนรู้ที่จะแนะนำของดัง" และ Coverage/Novelty ทรุดลงเท่า TopPopular (ดูตาราง Valid-B) — นี่คือ popularity bias ที่ต้อง
#   คุมด้วย guardrail หรือ IPS (2.12) ไม่ใช่ปล่อยให้ตัวชี้วัด accuracy ตัดสินคนเดียว ; ลองตัด 'popularity' ออกจาก FEAT แล้วเทียบ
FEAT_CACHE = {}
def make_ltr_scorer(model, hist):
    def fn(u):
        key = (u, id(hist))
        if key not in FEAT_CACHE: FEAT_CACHE[key] = user_features(u, hist)
        return model.predict_proba(FEAT_CACHE[key])[:, 1] + pop_tb
    return fn
LTR_LR, LTR_GB = make_ltr_scorer(ltr_lr, train_hist), make_ltr_scorer(ltr_gb, train_hist)
# ข้อสังเกต: LTR ถูกฝึกด้วย Valid-A จึง 'ห้าม' วัดผลบน Valid-A อีก → วัดบน Valid-B (ขั้นถัดไป) และ Test เท่านั้น

In [ ]:
# ---------- ปรับเทียบความน่าจะเป็นของ LTR-GB ด้วย Platt บน Valid-B และวัด ECE (ตัวอย่าง 3.23) ----------
XB, yB, gB = build_ltr_dataset(users_B, relB, train_hist, seed=SEED + 1)
raw_gb = ltr_gb.predict_proba(XB)[:, 1]
ece_before, tab_before = expected_calibration_error(raw_gb, yB, n_bins=10)
platt = PlattCalibrator().fit(np.log(raw_gb / (1 - raw_gb + 1e-9) + 1e-9), yB)     # ปรับเทียบบน logit
cal = platt.predict_proba(np.log(raw_gb / (1 - raw_gb + 1e-9) + 1e-9))
ece_after, tab_after = expected_calibration_error(cal, yB, n_bins=10)
print(f"ECE ก่อนปรับเทียบ = {ece_before:.4f} · หลัง Platt = {ece_after:.4f}  (a={platt.a_:.3f}, b={platt.b_:.3f})")
show_df(tab_before, "ตารางรายถังก่อนปรับเทียบ (สัดส่วนถูกจริง vs ความมั่นใจเฉลี่ย)", fmt="{}")
fig, ax = plt.subplots(figsize=(4.5, 4)); ax.plot([0, 1], [0, 1], "--", color="grey")
for tab, lab in [(tab_before, f"ก่อน ECE={ece_before:.3f}"), (tab_after, f"หลัง Platt ECE={ece_after:.3f}")]:
    ax.plot(tab["ความมั่นใจเฉลี่ย"], tab["สัดส่วนถูกจริง"], "o-", label=lab)
ax.set_xlabel("confidence"); ax.set_ylabel("observed frequency"); ax.set_title("Reliability diagram (Valid-B)"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
# ข้อสังเกต: ป้ายกำกับของชุดนี้มาจาก negative sampling 1:4 → ความน่าจะเป็น 'ตามฐาน' คือ ~0.2 ไม่ใช่สัดส่วนจริงของการชอบ
#            การปรับเทียบจึงถูกต้องเฉพาะ 'ภายใต้สัดส่วนตัวอย่างนี้' — ต้องประกาศเมื่อนำ τ ไปใช้จริง

## การทดลองเปรียบเทียบบน Valid-B: โมเดลค่าตั้งต้น (Lab 3-2) vs โมเดลที่ปรับจูน vs โมเดลทำนายขั้นสุดท้าย

In [ ]:
vec0 = TextVectorizer(min_df=2, max_df=0.85); X0 = vec0.fit_transform(build_text(1, 2, 1, 0)); bm0 = BM25().fit(vec0._train_tokens); Z0 = LSA(k=100, seed=SEED).fit(X0).transform(X0)
def d_tfidf(u): return match_scores(X0, build_profile(X0, train_hist.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR)) + pop_tb
def d_bm25(u):
    p = build_profile(X0, train_hist.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR); top = vec0.top_terms(p, 20)
    return bm0.score_query([t for t, _ in top], dict(top)) + pop_tb if top else None
def d_lsa(u): return match_scores(Z0, build_profile(Z0, train_hist.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR)) + pop_tb
def d_hybrid(u): return rrf_fusion([d_tfidf(u), s_genre(u), d_lsa(u)], k=60) + pop_tb
MODELS_B = {"TopPopular": lambda u: pop.p.copy(), "TFIDF (default)": d_tfidf, "TFIDF-Rocchio (tuned)": lambda u: s_tfidf(u) + pop_tb,
            "BM25 (default)": d_bm25, "BM25 (tuned)": lambda u: s_bm25(u) + pop_tb, "LSA (default k=100)": d_lsa, f"LSA (tuned k={BEST_LSA_K})": lambda u: s_lsa(u) + pop_tb,
            "NaiveBayes (tuned α)": lambda u: s_nb(u) + pop_tb, "Hybrid-RRF (default)": d_hybrid, f"Convex BM25/LSA (w={BEST_W})": lambda u: convex_fusion([s_bm25(u), s_lsa(u)], [BEST_W, 1 - BEST_W]) + pop_tb,
            "Hybrid-RRF (tuned)": s_hybrid, "LTR-LogReg (stacking)": LTR_LR, "LTR-GradBoost (stacking)": LTR_GB}
EXTRA_B = {f"Hybrid+MMR(λ={BEST_LAM})": (s_hybrid, make_mmr(BEST_LAM)), f"LTR-GB+MMR(λ={BEST_LAM})": (LTR_GB, make_mmr(BEST_LAM))}
summ_B, pu_B, recs_B = {}, {}, {}
for name, fn in MODELS_B.items():
    with Timer(name): summ_B[name], pu_B[name], recs_B[name] = quick_eval(fn, users=users_B, rel=relB, grades=grB)
for name, (fn, rr) in EXTRA_B.items():
    with Timer(name): summ_B[name], pu_B[name], recs_B[name] = quick_eval(fn, users=users_B, rel=relB, grades=grB, rerank_fn=rr)
tbl_B = results_table(summ_B, K=K); show_df(tbl_B, f"Valid-B (ผู้ใช้ {len(users_B)} คน): ค่าตั้งต้น vs ปรับจูน vs LTR — Accuracy | Beyond-Accuracy")
plot_metric_bars(tbl_B, [f"nDCG@{K}", f"HR@{K}", f"MRR@{K}", "AUC"], title="Accuracy on Valid-B", ncols=4)
plot_metric_bars(tbl_B, [f"Coverage@{K}", "Gini", f"LongTail@{K}", f"ILD@{K}", f"Novelty@{K}", "CalibErr"], title="Beyond-accuracy on Valid-B", ncols=3)
plot_tradeoff(tbl_B, f"nDCG@{K}", f"Coverage@{K}", "nDCG vs Coverage (Valid-B)")
# ข้อสังเกต: ดูว่า 'tuned' ชนะ 'default' บน Valid-B (ชุดที่ไม่ได้ใช้ปรับจูน) หรือไม่ — ถ้าไม่ชนะ = ปรับจูนพอดีกับสัญญาณรบกวนของ Valid-A

In [ ]:
# ---------- นัยสำคัญบน Valid-B: paired bootstrap เทียบกับ TFIDF (default) + Holm ----------
BASE = "TFIDF (default)"; rows, pv, nm = [], [], []
for name in tbl_B.index:
    if name == BASE: continue
    a = pu_B[BASE].set_index("user")["nDCG@K"]; b = pu_B[name].set_index("user")["nDCG@K"].reindex(a.index)
    bs = paired_bootstrap(a.values, b.values, 2000, SEED); rows.append({"โมเดล": name, "Δ nDCG@10": bs["diff"], "CI ล่าง": bs["ci_low"], "CI บน": bs["ci_high"], "p": bs["p_value"]}); pv.append(bs["p_value"])
boot = pd.DataFrame(rows).set_index("โมเดล"); boot["Holm ✔"] = ["✔" if r else "" for r in holm_bonferroni(pv)]
show_df(boot, f"paired bootstrap เทียบ {BASE} บน Valid-B")

## ขั้น ⑪ · โมเดลทำนายคะแนน (Rating): ปรับจูน Content-kNN (k · shrink · ปริภูมิ) และ Ridge บนคุณลักษณะ → MAE / RMSE

In [ ]:
rows = []
for space_name, M in [("TFIDF(tuned)", X), ("LSA", Z), ("Genre", G)]:
    for k, shrink in itertools.product([5, 10, 20, 50], [0.0, 5.0]):
        t0 = time.time(); m = ContentKNNRegressor(M, k=k, shrink=shrink).fit(train); p = np.clip(m.predict(valid_a), 0.5, 5)
        rows.append({"space": space_name, "k": k, "shrink": shrink, "MAE": mae(valid_a.r, p), "RMSE": rmse(valid_a.r, p), "วินาที": round(time.time() - t0, 1)})
res_knn = pd.DataFrame(rows); show_df(res_knn.sort_values("RMSE").head(10), "⑪ Content-kNN บน Valid-A (10 อันดับ RMSE ต่ำสุด)")
bk = res_knn.loc[res_knn["RMSE"].idxmin()]; BEST_SPACE, BEST_KK, BEST_SHRINK = bk["space"], int(bk["k"]), float(bk["shrink"])
SPACE_M = {"TFIDF(tuned)": X, "LSA": Z, "Genre": G}[BEST_SPACE]
print(f"เลือก space={BEST_SPACE}, k={BEST_KK}, shrink={BEST_SHRINK}")
fig, ax = plt.subplots(figsize=(6, 3.5))
for sp_, g in res_knn[res_knn.shrink == 0].groupby("space"): ax.plot(g["k"], g["RMSE"], "o-", label=sp_)
ax.set_xlabel("k"); ax.set_ylabel("RMSE (Valid-A)"); ax.legend(); ax.grid(alpha=.3); ax.set_title("Content-kNN: k × space"); plt.tight_layout(); plt.show()

# Ridge บนคุณลักษณะ (u,i): ค่าเฉลี่ยผู้ใช้ · ค่าเฉลี่ยรายการ · คะแนนจับคู่ (Rocchio/genre/LSA/BM25) · ความนิยม · ปี — โมเดลทำนายคะแนนแบบ "เรียนรู้"
from sklearn.linear_model import Ridge
def rating_features(pairs, hist, base):
    """คุณลักษณะต่อคู่ — base = MeanRatingBaseline ที่ fit บนชุดฝึก ; คะแนนจับคู่คำนวณต่อผู้ใช้แล้วดึงเฉพาะรายการในคู่"""
    feats, cache = [], {}
    for u, i in zip(pairs.u, pairs.i):
        if u not in cache:
            cache[u] = {"tfidf": s_tfidf(u), "genre": s_genre(u), "lsa": s_lsa(u), "bm25": minmax(s_bm25(u))} if u in hist else None
        c = cache[u]
        feats.append([base.mu + base.bu.get(u, 0), base.mu + base.bi.get(i, 0), *( [c["tfidf"][i], c["genre"][i], c["lsa"][i], c["bm25"][i]] if c else [0, 0, 0, 0]), pop.p[i], year_norm[i]])
    return np.array(feats)
base_ui = MeanRatingBaseline("user_item").fit(train)
with Timer("Ridge features"):
    FA, FB = rating_features(valid_a, train_hist, base_ui), rating_features(valid_b, train_hist, base_ui)
ridge = Ridge(alpha=1.0).fit(FA, valid_a.r)      # ฝึกบน Valid-A (คุณลักษณะจาก train) → ประเมินบน Valid-B
rating_models = {"GlobalMean": MeanRatingBaseline("global").fit(train), "UserMean": MeanRatingBaseline("user").fit(train), "User+ItemBias": base_ui,
                 "ContentKNN (default TFIDF k=20)": ContentKNNRegressor(X0, k=20).fit(train), f"ContentKNN (tuned {BEST_SPACE} k={BEST_KK})": ContentKNNRegressor(SPACE_M, k=BEST_KK, shrink=BEST_SHRINK).fit(train)}
pred_B = {n: np.clip(m.predict(valid_b), 0.5, 5) for n, m in rating_models.items()}; pred_B["Ridge (stacked features)"] = np.clip(ridge.predict(FB), 0.5, 5)
tbl_rB = pd.DataFrame({n: {"MAE": mae(valid_b.r, p), "RMSE": rmse(valid_b.r, p)} for n, p in pred_B.items()}).T
show_df(tbl_rB, f"MAE / RMSE บน Valid-B ({len(valid_b)} คู่)"); plot_metric_bars(tbl_rB, ["MAE", "RMSE"], title="Rating prediction (Valid-B)", ncols=2)
show_df(pd.DataFrame({"น้ำหนัก Ridge": ridge.coef_}, index=["user_mean", "item_mean", "tfidf", "genre", "lsa", "bm25", "popularity", "year"]).T, "น้ำหนักของ Ridge — คุณลักษณะใดช่วยทำนายคะแนน")

## ขั้น ⑫ · ทดสอบครั้งเดียวบน Test (ฝึกใหม่ด้วย train + valid-A + valid-B) · ตัวอย่างผลลัพธ์ · artifacts · GUI

In [ ]:
trainval = pd.concat([train, valid_a, valid_b]); hist_tv = hist_from(trainval)
pop_tv = ItemPopularity(trainval, N_ITEMS, N_USERS); head_tv = head_items_from_train(pop_tv.count, N_ITEMS); POOL_TV = set(int(i) for i in trainval.i.unique())
pop_tb = 1e-6 * minmax(pop_tv.p)
relT, grT = relevant_from(test, LIKE_THR); users_T = sorted(u for u, s in relT.items() if s and u in hist_tv)
# สร้าง scorer ใหม่ที่ผูกกับ hist_tv (ค่าไฮเพอร์พารามิเตอร์คงเดิม — ไม่ปรับอะไรอีก)
def f_tfidf(u): return match_scores(X, build_profile(X, hist_tv.get(u, {}), "rocchio", LIKE_THR, DISLIKE_THR, beta=BEST_BETA, gamma=BEST_GAMMA))
def f_genre(u): return match_scores(Gn, build_profile(G, hist_tv.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR))
def f_lsa(u):   return match_scores(Z, build_profile(Z, hist_tv.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR))
def f_bm25(u):
    p = build_profile(X, hist_tv.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR); top = VEC.top_terms(p, BEST_NQ)
    return BM.score_query([t for t, _ in top], dict(top)) if top else np.zeros(N_ITEMS)
def f_nb(u):
    s = ClassifierProfile("nb", alpha=BEST_ALPHA, like_thr=LIKE_THR, dislike_thr=DISLIKE_THR, min_examples=3).fit(C_counts, hist_tv.get(u, {})).score(C_counts)
    return s if s is not None else f_tfidf(u)
def f_hybrid(u): return rrf_fusion([f_bm25(u), f_lsa(u), f_genre(u), f_nb(u)], k=BEST_RRF_K) + pop_tb
BASE_SCORERS = {"tfidf_rocchio": f_tfidf, "genre": f_genre, "lsa": f_lsa, "bm25": f_bm25, "nb": f_nb}; pop = pop_tv
# LTR ฝึกใหม่ด้วยป้ายกำกับจาก Valid-A + Valid-B (คุณลักษณะจาก train+valid) — ยังไม่แตะ Test
FEAT_CACHE.clear()
relAB = {u: relA.get(u, set()) | relB.get(u, set()) for u in set(relA) | set(relB)}
XAB, yAB, _ = build_ltr_dataset(sorted(u for u, s in relAB.items() if s and u in hist_tv), relAB, hist_tv)
ltr_gb_final = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=4, random_state=SEED).fit(XAB, yAB)
FINAL_LTR = make_ltr_scorer(ltr_gb_final, hist_tv)
MODELS_T = {"TopPopular": lambda u: pop_tv.p.copy(), "TFIDF (default)": lambda u: match_scores(X0, build_profile(X0, hist_tv.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR)) + pop_tb,
            "BM25 (tuned)": lambda u: f_bm25(u) + pop_tb, "NaiveBayes (tuned α)": lambda u: f_nb(u) + pop_tb, "Hybrid-RRF (tuned)": f_hybrid, "LTR-GradBoost (final)": FINAL_LTR}
EXTRA_T = {f"LTR-GB+MMR(λ={BEST_LAM})": (FINAL_LTR, make_mmr(BEST_LAM, hist_tv, POOL_TV))}
summ_T, pu_T, recs_T = {}, {}, {}
for name, fn in MODELS_T.items():
    with Timer("test · " + name): summ_T[name], pu_T[name], recs_T[name] = quick_eval(fn, users=users_T, rel=relT, grades=grT, hist=hist_tv, pool=POOL_TV, pop_=pop_tv, head_=head_tv)
for name, (fn, rr) in EXTRA_T.items():
    with Timer("test · " + name): summ_T[name], pu_T[name], recs_T[name] = quick_eval(fn, users=users_T, rel=relT, grades=grT, hist=hist_tv, pool=POOL_TV, pop_=pop_tv, head_=head_tv, rerank_fn=rr)
tbl_T = results_table(summ_T, K=K); show_df(tbl_T, f"ผลบนชุดทดสอบ (ผู้ใช้ {len(users_T)} คน) — รายงานครั้งเดียว")
plot_metric_bars(tbl_T, [f"nDCG@{K}", f"HR@{K}", f"Coverage@{K}", f"Novelty@{K}", f"ILD@{K}", "Gini"], title="Test set (one-shot)", ncols=3)
a = pu_T["TopPopular"].set_index("user")["nDCG@K"]
for name in tbl_T.index:
    if name == "TopPopular": continue
    b = pu_T[name].set_index("user")["nDCG@K"].reindex(a.index); bs = paired_bootstrap(a.values, b.values, seed=SEED)
    print(f"{name:28s} Δ nDCG@{K} vs TopPopular = {bs['diff']:+.4f}  CI95 [{bs['ci_low']:+.4f}, {bs['ci_high']:+.4f}]  p={bs['p_value']:.3f}")
# Rating บน Test
knn_final = ContentKNNRegressor(SPACE_M, k=BEST_KK, shrink=BEST_SHRINK).fit(trainval); base_tv = MeanRatingBaseline("user_item").fit(trainval)
tbl_rT = pd.DataFrame({n: {"MAE": mae(test.r, p), "RMSE": rmse(test.r, p)} for n, p in {"UserMean": np.clip(MeanRatingBaseline("user").fit(trainval).predict(test), .5, 5),
              "User+ItemBias": np.clip(base_tv.predict(test), .5, 5), f"ContentKNN (tuned)": np.clip(knn_final.predict(test), .5, 5)}.items()}).T
show_df(tbl_rT, f"MAE / RMSE บนชุดทดสอบ ({len(test)} คู่)")

In [ ]:
# ---------- ตัวอย่างผลลัพธ์บนชุดทดสอบ: ประวัติ · ของจริง · Top-10 ของแต่ละโมเดล (✔) และคำอธิบาย ----------
def content_of(i): return movies.genres[i] + (f" · {movies.tag_text[i][:40]}" if movies.tag_text[i] else "")
hist_len = trainval.groupby("u").size(); rng = np.random.default_rng(SEED)
for label, u in [("ประวัติยาว", max(users_T, key=lambda u: hist_len.get(u, 0))), ("ประวัติสั้น", min(users_T, key=lambda u: hist_len.get(u, 0))), ("สุ่ม", int(rng.choice(users_T)))]:
    rep = user_report(u, hist_tv, test, recs_T, title_of, content_of, K=K, like_thr=LIKE_THR)
    print(f"\n{'='*25} ผู้ใช้ {u} ({label}) · ประวัติ {len(hist_tv[u])} เรื่อง · ของจริง ≥4 ในชุดทดสอบ {len(relT[u])} เรื่อง {'='*25}")
    show_df(rep["history"].head(6), "① ประวัติ (6 เรื่องคะแนนสูงสุด)", fmt="{}"); show_df(rep["ground_truth"].head(8), "② ของจริงในชุดทดสอบ", fmt="{}"); show_df(rep["recommended"], "③ Top-10 ของแต่ละโมเดล", fmt="{}")
    F = FEAT_CACHE.get((u, id(hist_tv)), user_features(u, hist_tv)); top1 = recs_T["LTR-GradBoost (final)"][u][0]
    print("   คุณลักษณะของอันดับ 1 ของ LTR:", title_of(top1), dict(zip(FEAT_NAMES, np.round(F[top1], 3))))

In [ ]:
# ---------- บันทึก artifacts: ตารางผล · รายการแนะนำ · ค่าที่เลือก · โมเดล ----------
import pickle
tbl_B.to_csv(f"{ART_DIR}/results_validB.csv"); tbl_T.to_csv(f"{ART_DIR}/results_test.csv"); tbl_rB.to_csv(f"{ART_DIR}/rating_validB.csv"); tbl_rT.to_csv(f"{ART_DIR}/rating_test.csv")
pd.DataFrame([{"user": u, "model": m, "rank": r + 1, "item": title_of(i), "hit": i in relT.get(u, set())} for m, rc in recs_T.items() for u, lst in rc.items() for r, i in enumerate(lst)]).to_csv(f"{ART_DIR}/recommendations_test.csv", index=False)
pickle.dump({"vectorizer": VEC, "bm25": BM, "lsa_Z": Z, "ltr": ltr_gb_final, "platt": platt, "chosen": CHOSEN, "knn": knn_final}, open(f"{ART_DIR}/models.pkl", "wb"))
print("บันทึกแล้ว:", sorted(os.listdir(ART_DIR)))
ProtocolCard({"1 เป้าหมายผู้ใช้/ธุรกิจ": "พบเรื่องที่จะให้ ≥4 ใน 10 อันดับแรก · เพิ่ม Coverage/Novelty โดยไม่ลด nDCG เกิน 5%", "2 Primary Metric": f"nDCG@{K}",
              "3 Guardrail Metrics": f"nDCG หลัง MMR ≥ 95% · Coverage@{K}, Gini, ILD, Novelty รายงานทุกโมเดล", "4 สูตรและค่า K / นิยามความเกี่ยวข้อง": f"K={K} · rel = rating ≥ {LIKE_THR} · {CHOSEN}",
              "5 การแบ่งข้อมูล": f"{K_CORE}-core → per-user temporal: train {len(train)} · valid-A {len(valid_a)} (ปรับจูนบน {len(users_tune)} ผู้ใช้) · valid-B {len(valid_b)} · test {len(test)}",
              "6 การป้องกันรั่วไหล": "สถิติทุกตัวจาก train · LTR ฝึกด้วย valid-A (ประเมินบน valid-B/test เท่านั้น) · Platt บน valid-B · test แตะครั้งเดียว",
              "7 Candidate Set": f"pool = รายการในชุดฝึก ({len(POOL_TV)} เรื่อง) หักที่เห็น · ตัวตัดสินรอง = ความนิยม", "8 Negative Sampling": "ประเมิน: ไม่ใช้ · ฝึก LTR: 1:4 จากรายการที่ไม่เคยเห็นใน pool",
              "9 Accuracy + Beyond-Accuracy": f"Test: " + " · ".join(f"{n} nDCG {tbl_T.loc[n, f'nDCG@{K}']:.4f}/Cov {tbl_T.loc[n, f'Coverage@{K}']:.3f}" for n in tbl_T.index[:4]),
              "10 ความไม่แน่นอน": f"paired bootstrap 2,000 รอบ + Holm บน Valid-B · CI บน Test ข้างต้น", "11 ตรวจกับผู้ใช้จริง": "ต้อง A/B test พร้อม guardrail เชิงธุรกิจ"}).show()

## ฟอร์ม GUI · สาธิตการประยุกต์ใช้โมเดลที่ปรับจูนแล้ว
แสดง **ข้อมูลผู้ใช้** (ประวัติ · คำเด่นในโปรไฟล์ · ของจริงในชุดทดสอบ) และ **รายการที่แนะนำ** พร้อม P(ชอบ) หลังปรับเทียบ · คุณลักษณะ · เหตุผลรายคำ ·
และ **คะแนนที่คาดว่าจะให้** (Content-kNN) · เลือกเกณฑ์ τ เพื่อแสดงเฉพาะรายการที่ P(ชอบ) ≥ τ

In [ ]:
inv_u = {v: k for k, v in u_map.items()}
GUI_MODELS = {**MODELS_T, f"LTR-GB+MMR(λ={BEST_LAM})": None}
def gui_user_info(user_id):
    u = u_map[user_id]; h = hist_tv.get(u, {})
    p = build_profile(X, h, "centroid", LIKE_THR, DISLIKE_THR)
    print(f"ประวัติ {len(h)} เรื่อง · คำเด่นในโปรไฟล์: " + ", ".join(f"{t} {w}" for t, w in VEC.top_terms(p, 10)))
    print("ของจริงในชุดทดสอบ (≥4):", ", ".join(title_of(i) for i in list(relT.get(u, []))[:8]))
    return pd.DataFrame([{"รายการ": title_of(i), "คะแนน": r, "แนว": movies.genres[i], "tags": movies.tag_text[i][:40]} for i, r in sorted(h.items(), key=lambda x: -x[1])[:10]])
def gui_recommend(user_id, model_name, Kk, tau=0.0):
    u = u_map[user_id]; h = hist_tv.get(u, {})
    if model_name.startswith("LTR-GB+MMR"):
        s = FINAL_LTR(u); rec = make_mmr(BEST_LAM, hist_tv, POOL_TV)(u, s)[:Kk]
    else:
        s = GUI_MODELS[model_name](u); rec = top_k(s, set(h), Kk, candidates=POOL_TV)
    F = FEAT_CACHE.get((u, id(hist_tv)), user_features(u, hist_tv)); raw = ltr_gb_final.predict_proba(F)[:, 1]
    p_cal = platt.predict_proba(np.log(raw / (1 - raw + 1e-9) + 1e-9))
    prof = build_profile(X, h, "centroid", LIKE_THR, DISLIKE_THR); rows = []
    for rank, i in enumerate(rec, 1):
        if p_cal[i] < tau: continue
        rows.append({"อันดับ": rank, "ตรงของจริง": "✔" if i in relT.get(u, set()) else "", "รายการ": title_of(i), "คะแนนโมเดล": round(float(s[i]), 4), "P(ชอบ) ปรับเทียบ": round(float(p_cal[i]), 3),
                     "คะแนนที่คาด (kNN)": round(float(np.clip(knn_final.predict_one(u, i), .5, 5)), 2), "แนว": movies.genres[i], "p(i)": round(float(pop_tv[i]), 3),
                     "คำร่วมกับโปรไฟล์": ", ".join(t for t, *_ in explain_terms(VEC, prof, X[i], 3)) or "—"})
    return pd.DataFrame(rows)
try:
    import ipywidgets as W
    w_tau = W.FloatSlider(value=0.0, min=0.0, max=0.9, step=0.05, description="τ P(ชอบ)≥", continuous_update=False)
    form = build_recommender_form([inv_u[u] for u in users_T], list(GUI_MODELS), lambda uid, m, k: gui_recommend(uid, m, k, w_tau.value), gui_user_info, K_default=K, K_max=30,
                                  title="MovieLens · Pipeline ที่ปรับจูนแล้ว (สาธิตการประยุกต์ใช้)")
    display(w_tau)
except Exception as e:
    print("GUI ใช้ไม่ได้:", e); display(gui_recommend(inv_u[users_T[0]], "LTR-GradBoost (final)", K))

## สรุปข้อสังเกตของ Lab 3-3

1. **ลำดับการปรับจูนสำคัญกว่าขนาดกริด** — น้ำหนักเขตข้อมูลและการคุมคลังศัพท์ (ลำดับ 1–3) มักย้ายผลมากกว่าพารามิเตอร์ของฟังก์ชันจัดอันดับ (ลำดับ 5)
2. **สองชุดตรวจสอบ** ทำให้เห็นว่าค่าที่เลือกจาก Valid-A ยังชนะบน Valid-B หรือไม่ — ถ้าไม่ คือ "ปรับให้พอดีกับสัญญาณรบกวน"
3. **LTR แบบ stacking** ใช้คะแนนของทุกระบบเป็นคุณลักษณะ จึงมักได้ accuracy สูงสุด แต่ต้องระวังว่าคุณลักษณะ 'ความนิยม' อาจครอบงำ (ดูน้ำหนัก LR) และลด Coverage
4. **การปรับเทียบ (Platt/ECE)** ทำให้ตั้งเกณฑ์ τ ได้ — แต่ความน่าจะเป็นถูกต้องภายใต้สัดส่วน negative sampling ที่ใช้ฝึกเท่านั้น
5. **MMR ภายใต้ guardrail** เปลี่ยนคำถามจาก "λ ไหนดีที่สุด" เป็น "ยอมเสีย nDCG เท่าไรเพื่อ ILD เท่าไร" ซึ่งเป็นการตัดสินใจเชิงผลิตภัณฑ์
6. **Rating กับ Ranking ใช้โมเดลคนละตัว** — Content-kNN ที่ RMSE ดีที่สุดไม่ใช่ตัวที่จัดอันดับดีที่สุด และ Ridge ที่รวมคุณลักษณะช่วย RMSE ได้เพิ่ม
7. **ข้อจำกัด** — เนื้อหาของ MovieLens บาง (ไม่มีเรื่องย่อ) · ปรับจูนบนผู้ใช้กลุ่มย่อย · ไม่มี Sentence-Embedding จริงในเล่มนี้ (ใส่แทน LSA ได้ทันทีด้วย `sentence_embeddings`)